In [5]:
# ============================================================
# ONE-CELL NOTEBOOK RUNNER (PROC)
# - signal timeframe: 1h (strategy confirmations)
# - execution timeframe: 5m (hit-time TP/SL)
# - grid-search over TP/SL using trade-list evaluation
# ============================================================

import os
import re
import math
import time
import numpy as np
import pandas as pd
import numba as nb
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


def njit_cached(
    *,
    parallel: bool = False,
    fastmath: bool = False,
    inline: str = "never",
):
    """Return `numba.njit` decorator with cache fallback for notebook execution.

    Parameters:
        parallel: Enable Numba parallel transformation.
        fastmath: Enable relaxed floating-point optimizations.
        inline: Numba inline policy.

    Returns:
        Callable decorator that first tries `cache=True`, then falls back to
        `cache=False` only when Numba reports missing file locator.

    Assumptions:
        Notebook/interactive execution may not provide a cache locator.

    Raises:
        RuntimeError: Re-raises non-locator Numba runtime errors.
    """

    def decorate(func):
        try:
            return nb.njit(
                parallel=parallel,
                fastmath=fastmath,
                inline=inline,
                cache=True,
            )(func)
        except RuntimeError as exc:
            if "no locator available" not in str(exc).lower():
                raise
            return nb.njit(
                parallel=parallel,
                fastmath=fastmath,
                inline=inline,
                cache=False,
            )(func)

    return decorate

# -------------------------
# 0) Paths
# -------------------------
BASE_DIR = "/Users/daniildegtyarev/Projects/roehub.com/tests/notebook_tests/precompute/btcusdt_5m"
NPY_PATH = os.path.join(BASE_DIR, "prices_and_signals_5m.npy")

HIT_LONG_TP_PATH = os.path.join(BASE_DIR, "btc_5m.long_tp.u32.npy")
HIT_LONG_SL_PATH = os.path.join(BASE_DIR, "btc_5m.long_sl.u32.npy")
HIT_SHORT_TP_PATH = os.path.join(BASE_DIR, "btc_5m.short_tp.u32.npy")
HIT_SHORT_SL_PATH = os.path.join(BASE_DIR, "btc_5m.short_sl.u32.npy")

USE_MMAP = True

# -------------------------
# 1) Config
# -------------------------
signal_tf = "1h"
exec_tf = "5m"
fee_rate = 0.0004
close_on_end = 1

# Pre-filters (E + A/B/D)
top_frac_side = math.sqrt(0.10)
min_confirm = 30
top_frac_pairs = 0.10
time_chunk = 4096
min_nonzero_single = 200

# TP/SL grid (must match precompute)
tp_start_pct, tp_stop_pct, tp_step_pct = 4.0, 50.0, 0.5
sl_start_pct, sl_stop_pct, sl_step_pct = 2.0, 25.0, 0.5

USE_PRECOMPUTED_SIGNALS = int(os.environ.get("BT_USE_PRECOMPUTED_SIGNALS", "1")) == 1

print("BASE_DIR:", BASE_DIR)
print("NPY:", NPY_PATH)
print(
    "HIT files exist:",
    os.path.exists(HIT_LONG_TP_PATH),
    os.path.exists(HIT_LONG_SL_PATH),
    os.path.exists(HIT_SHORT_TP_PATH),
    os.path.exists(HIT_SHORT_SL_PATH),
)

print("\nConfig:")
print("  signal_tf:", signal_tf, "exec_tf:", exec_tf)
print("  fee_rate:", fee_rate, "close_on_end:", close_on_end)
print("  E: top_frac_side:", top_frac_side, "min_nonzero_single:", min_nonzero_single)
print("  A/B/D: min_confirm:", min_confirm, "top_frac_pairs:", top_frac_pairs, "time_chunk:", time_chunk)
print("  TP grid:", (tp_start_pct, tp_stop_pct, tp_step_pct))
print("  SL grid:", (sl_start_pct, sl_stop_pct, sl_step_pct))


# -------------------------
# 2) Load data + columns
# -------------------------
def _load_columns_or_fail(base_dir: str, expected_len: int) -> tuple[str, ...]:
    """Load ndarray column names from globals or from a sidecar file.

    Parameters:
        base_dir: Directory containing precompute artifacts.
        expected_len: Expected number of columns in the ndarray.

    Returns:
        Tuple of column names in ndarray order.

    Assumptions:
        A columns sidecar file exists in `base_dir` when globals are absent.

    Raises:
        RuntimeError: If no compatible columns definition can be found.
    """
    if "prices_and_signals_np_columns" in globals():
        cols = globals()["prices_and_signals_np_columns"]
        if isinstance(cols, (list, tuple)) and len(cols) == expected_len:
            return tuple(cols)

    candidates = []
    for fn in os.listdir(base_dir):
        low = fn.lower()
        if (
            ("column" in low or "columns" in low or "cols" in low)
            and (
                low.endswith(".npy")
                or low.endswith(".pkl")
                or low.endswith(".pickle")
                or low.endswith(".json")
            )
        ):
            candidates.append(os.path.join(base_dir, fn))
    candidates = sorted(candidates)

    for path in candidates:
        low = path.lower()
        try:
            if low.endswith(".npy"):
                cols = np.load(path, allow_pickle=True)
                cols = tuple(cols.tolist()) if hasattr(cols, "tolist") else tuple(cols)
            elif low.endswith(".pkl") or low.endswith(".pickle"):
                import pickle

                with open(path, "rb") as f:
                    cols = tuple(pickle.load(f))
            elif low.endswith(".json"):
                import json

                with open(path, "r", encoding="utf-8") as f:
                    cols = tuple(json.load(f))
            else:
                continue
            if len(cols) == expected_len:
                print("Loaded columns from:", path)
                return cols
        except Exception:
            pass

    raise RuntimeError(
        "Cannot locate column names. Provide `prices_and_signals_np_columns` "
        "or store a columns sidecar in BASE_DIR (e.g. prices_and_signals_5m_columns.npy)."
    )


_TF_RE = re.compile(r"^\s*(\d+)\s*([mhd])\s*$", re.IGNORECASE)


def timeframe_to_bars_per_year(tf: str) -> float:
    """Convert timeframe string to bars-per-year constant.

    Parameters:
        tf: Timeframe token like `5m`, `1h`, or `1d`.

    Returns:
        Number of bars in 365 days for the given timeframe.

    Raises:
        ValueError: If timeframe format is unsupported.
    """
    m = _TF_RE.match(tf)
    if not m:
        raise ValueError(f"Bad timeframe: {tf!r}")
    n = int(m.group(1))
    unit = m.group(2).lower()
    minutes = {"m": n, "h": n * 60, "d": n * 1440}[unit]
    return (365.0 * 24.0 * 60.0) / minutes


def bars_per_year_from_dt_ms(dt_ms: int) -> float:
    """Convert bar duration in milliseconds to bars-per-year.

    Parameters:
        dt_ms: Bar duration in milliseconds.

    Returns:
        Number of bars in 365 days for the given duration.

    Raises:
        ValueError: If `dt_ms` is not positive.
    """
    if dt_ms <= 0:
        raise ValueError(f"dt_ms must be positive, got {dt_ms}")
    return (365.0 * 24.0 * 60.0 * 60.0 * 1000.0) / float(dt_ms)


def infer_median_positive_dt_ms(open_time_ms: np.ndarray) -> int:
    """Infer effective bar duration from median positive open-time delta.

    Parameters:
        open_time_ms: Monotonic open timestamps in milliseconds.

    Returns:
        Median positive delta in milliseconds.

    Raises:
        ValueError: If fewer than two bars or no positive deltas exist.
    """
    if open_time_ms.shape[0] < 2:
        raise ValueError("Need at least two bars to infer timeframe.")
    diff = np.diff(open_time_ms.astype(np.int64, copy=False))
    diff = diff[diff > 0]
    if diff.size == 0:
        raise ValueError("Cannot infer timeframe: no positive open_time deltas.")
    return int(np.median(diff))


def timeframe_label_from_dt_ms(dt_ms: int) -> str:
    """Map known millisecond durations to canonical timeframe labels.

    Parameters:
        dt_ms: Bar duration in milliseconds.

    Returns:
        Human-readable timeframe label.
    """
    known = {
        60_000: "1m",
        300_000: "5m",
        900_000: "15m",
        1_800_000: "30m",
        3_600_000: "1h",
        14_400_000: "4h",
        86_400_000: "1d",
    }
    if dt_ms in known:
        return known[dt_ms]
    if dt_ms % 60_000 == 0:
        return f"{dt_ms // 60_000}m"
    return f"{dt_ms}ms"


def positive_dt_stats_ms(open_time_ms: np.ndarray) -> tuple[int, int, int, int]:
    """Return median/min/max positive open-time delta and non-positive count.

    Parameters:
        open_time_ms: Open timestamps in milliseconds.

    Returns:
        Tuple `(median_positive_dt_ms, min_positive_dt_ms, max_positive_dt_ms, non_positive_count)`.

    Assumptions:
        Timestamp array represents bar opens in chronological order.

    Raises:
        ValueError: If fewer than two bars or no positive deltas exist.
    """
    if open_time_ms.shape[0] < 2:
        raise ValueError("Need at least two bars to infer timeframe.")

    diff = np.diff(open_time_ms.astype(np.int64, copy=False))
    non_positive_count = int((diff <= 0).sum())
    positive = diff[diff > 0]
    if positive.size == 0:
        raise ValueError("Cannot infer timeframe: no positive open_time deltas.")

    return (
        int(np.median(positive)),
        int(np.min(positive)),
        int(np.max(positive)),
        non_positive_count,
    )


def pct_grid(start_pct: float, stop_pct: float, step_pct: float) -> np.ndarray:
    """Create an inclusive percent grid with float32 dtype.

    Parameters:
        start_pct: Start value in percent.
        stop_pct: Stop value in percent.
        step_pct: Step value in percent.

    Returns:
        Float32 ndarray with inclusive stop handling.

    Raises:
        ValueError: If produced grid is empty.
    """
    eps = 1e-9
    grid = np.arange(start_pct, stop_pct + eps, step_pct, dtype=np.float32)
    if grid.size == 0:
        raise ValueError("Empty grid")
    return grid


def parse_signal_column_specs(signal_cols: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """Parse signal column names into source and window arrays.

    Parameters:
        signal_cols: Column names with format `signal|ma.<kind>|...|source=...|window=...`.

    Returns:
        Tuple `(sources, windows)` where:
        - `sources` is object ndarray of source tokens;
        - `windows` is int32 ndarray of MA windows.

    Assumptions:
        Each name includes `source=` and `window=` key-value tokens.

    Raises:
        ValueError: If required tokens are missing or malformed.
    """
    sources = []
    windows = []

    for name in signal_cols:
        parts = str(name).split("|")
        params = {}
        for part in parts[2:]:
            if "=" in part:
                key, value = part.split("=", 1)
                params[key] = value

        source = params.get("source")
        window_raw = params.get("window")
        if source is None or window_raw is None:
            raise ValueError(f"Cannot parse source/window from signal column: {name!r}")

        try:
            window = int(window_raw)
        except ValueError as exc:
            raise ValueError(f"Bad window in signal column: {name!r}") from exc

        if window <= 0:
            raise ValueError(f"Window must be positive in signal column: {name!r}")

        sources.append(source)
        windows.append(window)

    return np.array(sources, dtype=object), np.array(windows, dtype=np.int32)


def aggregate_5m_to_1h(
    open_time_ms: np.ndarray,
    open_px: np.ndarray,
    high_px: np.ndarray,
    low_px: np.ndarray,
    close_px: np.ndarray,
    volume: np.ndarray,
) -> dict[str, np.ndarray]:
    """Aggregate 5m OHLCV arrays into deterministic 1h OHLCV arrays.

    Parameters:
        open_time_ms: 5m bar open timestamps in milliseconds (ascending).
        open_px: 5m open prices.
        high_px: 5m high prices.
        low_px: 5m low prices.
        close_px: 5m close prices.
        volume: 5m volume values.

    Returns:
        Dict with 1h arrays: `open_time_ms`, `close_time_ms`, `open`, `high`,
        `low`, `close`, `volume`.

    Assumptions:
        Input arrays share the same length and chronological order.

    Side effects:
        None.
    """
    hour_ms = np.int64(60 * 60 * 1000)
    hour_open = (open_time_ms // hour_ms) * hour_ms

    uniq_hours, first_idx, counts = np.unique(hour_open, return_index=True, return_counts=True)
    last_idx = first_idx + counts - 1

    out_open = open_px[first_idx]
    out_close = close_px[last_idx]
    out_high = np.maximum.reduceat(high_px, first_idx)
    out_low = np.minimum.reduceat(low_px, first_idx)
    out_volume = np.add.reduceat(volume, first_idx)
    out_close_time = uniq_hours + hour_ms - np.int64(1)

    return {
        "open_time_ms": uniq_hours.astype(np.int64, copy=False),
        "close_time_ms": out_close_time.astype(np.int64, copy=False),
        "open": out_open.astype(np.float64, copy=False),
        "high": out_high.astype(np.float64, copy=False),
        "low": out_low.astype(np.float64, copy=False),
        "close": out_close.astype(np.float64, copy=False),
        "volume": out_volume.astype(np.float64, copy=False),
    }


def build_source_arrays_from_ohlc(ohlc_1h: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Build indicator source arrays used by SMA/EMA signal definitions.

    Parameters:
        ohlc_1h: Aggregated 1h OHLCV dict.

    Returns:
        Mapping of source name to float64 ndarray.

    Assumptions:
        Supported sources are `open`, `high`, `low`, `close`, `hlc3`, `ohlc4`.
    """
    open_ = ohlc_1h["open"]
    high = ohlc_1h["high"]
    low = ohlc_1h["low"]
    close = ohlc_1h["close"]

    return {
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "hlc3": (high + low + close) / np.float64(3.0),
        "ohlc4": (open_ + high + low + close) / np.float64(4.0),
    }


@njit_cached()
def compute_sma_signal_matrix(source: np.ndarray, windows: np.ndarray) -> np.ndarray:
    """Compute SMA-based {-1,0,1} signals for many windows on one source series.

    Parameters:
        source: Float64 source series.
        windows: Int32 windows array.

    Returns:
        Int8 matrix with shape `(n_windows, n_bars)`.

    Assumptions:
        Window values are strictly positive.
    """
    n = source.shape[0]
    n_w = windows.shape[0]

    out = np.zeros((n_w, n), dtype=np.int8)
    csum = np.zeros(n + 1, dtype=np.float64)

    for i in range(n):
        csum[i + 1] = csum[i] + source[i]

    for wi in range(n_w):
        w = int(windows[wi])
        if w <= 0 or w > n:
            continue

        inv_w = 1.0 / float(w)
        start = w - 1

        for t in range(start, n):
            ma = (csum[t + 1] - csum[t + 1 - w]) * inv_w
            v = source[t]
            if v > ma:
                out[wi, t] = np.int8(1)
            elif v < ma:
                out[wi, t] = np.int8(-1)
            else:
                out[wi, t] = np.int8(0)

    return out


@njit_cached()
def compute_ema_signal_matrix(source: np.ndarray, windows: np.ndarray) -> np.ndarray:
    """Compute EMA-based {-1,0,1} signals for many windows on one source series.

    Parameters:
        source: Float64 source series.
        windows: Int32 windows array.

    Returns:
        Int8 matrix with shape `(n_windows, n_bars)`.

    Assumptions:
        EMA is initialized with first source value.
    """
    n = source.shape[0]
    n_w = windows.shape[0]

    out = np.zeros((n_w, n), dtype=np.int8)

    for wi in range(n_w):
        w = int(windows[wi])
        if w <= 0:
            continue

        alpha = 2.0 / (float(w) + 1.0)
        ema = source[0]

        for t in range(1, n):
            ema = alpha * source[t] + (1.0 - alpha) * ema
            v = source[t]
            if v > ema:
                out[wi, t] = np.int8(1)
            elif v < ema:
                out[wi, t] = np.int8(-1)
            else:
                out[wi, t] = np.int8(0)

    return out


def build_signal_matrix_for_specs(
    signal_cols: list[str],
    sources: np.ndarray,
    windows: np.ndarray,
    source_arrays: dict[str, np.ndarray],
    kind: str,
) -> np.ndarray:
    """Build full signal matrix in original column order for SMA/EMA specs.

    Parameters:
        signal_cols: Ordered signal column names.
        sources: Parsed source token per column.
        windows: Parsed window per column.
        source_arrays: Source vectors built from 1h OHLC.
        kind: Either `sma` or `ema`.

    Returns:
        Int8 matrix with shape `(len(signal_cols), n_1h_bars)`.

    Assumptions:
        All requested sources exist in `source_arrays`.

    Raises:
        ValueError: If `kind` is unsupported or source is missing.
    """
    if kind not in {"sma", "ema"}:
        raise ValueError(f"Unsupported kind: {kind!r}")

    unique_windows = np.array(sorted({int(w) for w in windows.tolist()}), dtype=np.int32)
    window_to_pos = {int(w): i for i, w in enumerate(unique_windows.tolist())}

    n_signals = len(signal_cols)
    n_bars = int(next(iter(source_arrays.values())).shape[0])
    out = np.zeros((n_signals, n_bars), dtype=np.int8)

    precomputed: dict[str, np.ndarray] = {}

    for src in sorted({str(s) for s in sources.tolist()}):
        if src not in source_arrays:
            raise ValueError(f"Unknown signal source: {src!r}")
        series = np.ascontiguousarray(source_arrays[src], dtype=np.float64)
        if kind == "sma":
            precomputed[src] = compute_sma_signal_matrix(series, unique_windows)
        else:
            precomputed[src] = compute_ema_signal_matrix(series, unique_windows)

    for i in range(n_signals):
        src = str(sources[i])
        w = int(windows[i])
        out[i, :] = precomputed[src][window_to_pos[w], :]

    return np.ascontiguousarray(out)


def topk_fraction_idx(score: np.ndarray, frac: float) -> np.ndarray:
    """Select top scoring indices by fraction.

    Parameters:
        score: 1D score array.
        frac: Fraction in (0, 1].

    Returns:
        1D array of selected indices.

    Assumptions:
        `score` is finite for candidates that should be selected.
    """
    n = score.shape[0]
    k = max(1, int(math.ceil(n * frac)))
    return np.argpartition(score, n - k)[n - k:]


def single_score_chunked(sig_T_i8: np.ndarray, ret_f32: np.ndarray, chunk: int) -> np.ndarray:
    """Compute chunked dot-product score for single-indicator prefilter.

    Parameters:
        sig_T_i8: Int8 signals with shape `(n_strategies, n_intervals)`.
        ret_f32: Float32 returns with shape `(n_intervals,)`.
        chunk: Chunk length along time axis.

    Returns:
        Float32 score per strategy.

    Assumptions:
        Shapes are aligned on `n_intervals`.
    """
    n_strat, n_int_ = sig_T_i8.shape
    out = np.zeros(n_strat, dtype=np.float32)
    for t0 in range(0, n_int_, chunk):
        t1 = min(t0 + chunk, n_int_)
        out += sig_T_i8[:, t0:t1].astype(np.float32) @ ret_f32[t0:t1]
    return out


mm = "r" if USE_MMAP else None
prices_and_signals_np = np.load(NPY_PATH, mmap_mode=mm, allow_pickle=False)
cols = _load_columns_or_fail(BASE_DIR, expected_len=prices_and_signals_np.shape[1])

arr = prices_and_signals_np
assert isinstance(arr, np.ndarray) and arr.ndim == 2
assert len(cols) == arr.shape[1]

col_to_i = {c: i for i, c in enumerate(cols)}

required = ["open_time", "close_time", "open", "high", "low", "close"]
missing = [c for c in required if c not in col_to_i]
if missing:
    raise KeyError(f"Missing required columns in columns sidecar: {missing}")

open_time_idx = col_to_i["open_time"]
close_time_idx = col_to_i["close_time"]
open_idx = col_to_i["open"]
high_idx = col_to_i["high"]
low_idx = col_to_i["low"]
close_idx = col_to_i["close"]
volume_idx = col_to_i.get("volume", -1)

sma_cols = [c for c in cols if isinstance(c, str) and c.startswith("signal|ma.sma|")]
ema_cols = [c for c in cols if isinstance(c, str) and c.startswith("signal|ma.ema|")]
signal_cols_available = bool(sma_cols) and bool(ema_cols)

print("\nData shape:", arr.shape)
print("SMA cols:", len(sma_cols), "EMA cols:", len(ema_cols))
if not signal_cols_available:
    print("Warning: SMA/EMA signal columns are missing; precomputed signal path is unavailable.")


# -------------------------
# Execution TF precompute loading
# -------------------------
hit_long_tp = np.load(HIT_LONG_TP_PATH, mmap_mode=mm, allow_pickle=False)
hit_long_sl = np.load(HIT_LONG_SL_PATH, mmap_mode=mm, allow_pickle=False)
hit_short_tp = np.load(HIT_SHORT_TP_PATH, mmap_mode=mm, allow_pickle=False)
hit_short_sl = np.load(HIT_SHORT_SL_PATH, mmap_mode=mm, allow_pickle=False)

if (
    hit_long_tp.dtype != np.uint32
    or hit_long_sl.dtype != np.uint32
    or hit_short_tp.dtype != np.uint32
    or hit_short_sl.dtype != np.uint32
):
    raise TypeError("Hit-time tables must be uint32 .npy arrays.")

T_exec = int(arr.shape[0])
if hit_long_tp.shape[1] != T_exec:
    raise ValueError(f"Hit-time T mismatch: hit has {hit_long_tp.shape[1]} bars, arr has {T_exec}")

hit_tables = {
    "long_tp": hit_long_tp,
    "long_sl": hit_long_sl,
    "short_tp": hit_short_tp,
    "short_sl": hit_short_sl,
}
for hit_name, hit_table in hit_tables.items():
    if hit_table.shape[0] > 1 and not np.all(hit_table[1:, :] >= hit_table[:-1, :]):
        raise RuntimeError(
            f"Hit-time table {hit_name!r} is not monotone across level axis. "
            "Fast monotone TP/SL kernel requires monotonicity."
        )

# Timeframe reality check (runtime normalization)
open_time_ms_raw = arr[:, open_time_idx].astype(np.int64, copy=False)
close_time_ms_raw = arr[:, close_time_idx].astype(np.int64, copy=False)
detected_dt_ms, exec_dt_min_ms, exec_dt_max_ms, exec_non_pos_deltas = positive_dt_stats_ms(open_time_ms_raw)
if exec_non_pos_deltas > 0:
    raise ValueError(
        "Execution open_time must be strictly increasing. "
        f"Found {exec_non_pos_deltas} non-positive deltas."
    )
effective_exec_tf = timeframe_label_from_dt_ms(detected_dt_ms)
effective_signal_tf = "unknown"
detected_signal_dt_ms = np.int64(-1)
signal_dt_min_ms = np.int64(-1)
signal_dt_max_ms = np.int64(-1)
signal_non_pos_deltas = np.int64(0)
is_native_1h_exec = abs(detected_dt_ms - 3_600_000) <= 1_000

# Keep user-facing intent, but use detected execution cadence for metrics.
bars_per_year_exec = float(bars_per_year_from_dt_ms(detected_dt_ms))

# Grid validation
tp_grid = pct_grid(tp_start_pct, tp_stop_pct, tp_step_pct)
sl_grid = pct_grid(sl_start_pct, sl_stop_pct, sl_step_pct)

n_tp = int(tp_grid.size)
n_sl = int(sl_grid.size)
G = n_tp * n_sl

if hit_long_tp.shape[0] != n_tp or hit_long_sl.shape[0] != n_sl:
    raise ValueError(
        "Grid mismatch with precompute:\n"
        f"  precompute TP levels={hit_long_tp.shape[0]}, config TP levels={n_tp}\n"
        f"  precompute SL levels={hit_long_sl.shape[0]}, config SL levels={n_sl}\n"
        "Adjust tp/sl ranges to match precompute."
    )

tp = (tp_grid / np.float32(100.0)).astype(np.float32)
sl = (sl_grid / np.float32(100.0)).astype(np.float32)

tp_mult_up = (np.float32(1.0) + tp).astype(np.float32)
tp_mult_down = (np.float32(1.0) - tp).astype(np.float32)
sl_mult_up = (np.float32(1.0) + sl).astype(np.float32)
sl_mult_down = (np.float32(1.0) - sl).astype(np.float32)

if np.any(tp_mult_down <= 0) or np.any(sl_mult_down <= 0):
    raise ValueError("Bad grid: (1 - tp) or (1 - sl) <= 0.")

long_tp_eq = tp_mult_up
long_sl_eq = sl_mult_down
short_tp_eq = tp_mult_up
short_sl_eq = sl_mult_down


fee_two_sides = float((1.0 - fee_rate) * (1.0 - fee_rate))
if fee_two_sides <= 0.0:
    raise ValueError("fee_rate produces a non-positive two-sided fee factor.")

log_fac_tp_long = np.ascontiguousarray(np.log(np.asarray(long_tp_eq, dtype=np.float64) * fee_two_sides))
log_fac_sl_long = np.ascontiguousarray(np.log(np.asarray(long_sl_eq, dtype=np.float64) * fee_two_sides))
log_fac_tp_short = np.ascontiguousarray(np.log(np.asarray(short_tp_eq, dtype=np.float64) * fee_two_sides))
log_fac_sl_short = np.ascontiguousarray(np.log(np.asarray(short_sl_eq, dtype=np.float64) * fee_two_sides))

print("\nHit-time loaded:")
print("  long_tp:", hit_long_tp.shape, hit_long_tp.dtype)
print("  long_sl:", hit_long_sl.shape, hit_long_sl.dtype)
print("  short_tp:", hit_short_tp.shape, hit_short_tp.dtype)
print("  short_sl:", hit_short_sl.shape, hit_short_sl.dtype)
print("  TP levels:", n_tp, "SL levels:", n_sl, "grid cells:", G)
print("  bars_per_year_exec:", bars_per_year_exec)

print("\nTimeframe reality check:")
print("  signal_tf_intent:", signal_tf)
print("  exec_tf_intent:", exec_tf)
print("  detected_exec_dt_ms:", detected_dt_ms)
print("  detected_exec_tf:", effective_exec_tf)
print("  exec_dt_min_ms:", exec_dt_min_ms, "exec_dt_max_ms:", exec_dt_max_ms)
if exec_dt_min_ms != exec_dt_max_ms:
    print("  warning: execution dt is not constant; median positive delta is used")
print("  effective_exec_tf:", effective_exec_tf)
if is_native_1h_exec:
    print("  mode: native-1h execution indexing (no 5m->1h aggregation)")
else:
    print("  mode: aggregated signal 1h + searchsorted mapping into execution bars")


# -------------------------
# Signal TF (1h) preparation
# -------------------------
print("\nSignal TF (1h) preparation ...")

use_precomputed_signals = USE_PRECOMPUTED_SIGNALS and signal_cols_available
if USE_PRECOMPUTED_SIGNALS and not signal_cols_available:
    print("  precomputed signal columns unavailable; using recomputation fallback")

signal_open_time_ms: np.ndarray
signal_close_time_ms: np.ndarray
close_signal: np.ndarray
sma_sig_full: np.ndarray
ema_sig_full: np.ndarray

if use_precomputed_signals:
    sma_col_idx = np.array([col_to_i[c] for c in sma_cols], dtype=np.int32)
    ema_col_idx = np.array([col_to_i[c] for c in ema_cols], dtype=np.int32)

    sma_is_block = bool(np.all(sma_col_idx == np.arange(int(sma_col_idx[0]), int(sma_col_idx[0]) + sma_col_idx.size)))
    ema_is_block = bool(np.all(ema_col_idx == np.arange(int(ema_col_idx[0]), int(ema_col_idx[0]) + ema_col_idx.size)))

    if sma_is_block:
        sma_start = int(sma_col_idx[0])
        sma_stop = sma_start + int(sma_col_idx.size)
        sma_sig_native = np.asarray(arr[:, sma_start:sma_stop])
    else:
        sma_sig_native = np.asarray(arr[:, sma_col_idx])

    if ema_is_block:
        ema_start = int(ema_col_idx[0])
        ema_stop = ema_start + int(ema_col_idx.size)
        ema_sig_native = np.asarray(arr[:, ema_start:ema_stop])
    else:
        ema_sig_native = np.asarray(arr[:, ema_col_idx])

    if not np.isfinite(sma_sig_native).all():
        raise ValueError("Precomputed SMA signals contain NaN/inf values.")
    if not np.isfinite(ema_sig_native).all():
        raise ValueError("Precomputed EMA signals contain NaN/inf values.")

    eps_sig = 1e-6
    sma_round = np.rint(sma_sig_native)
    ema_round = np.rint(ema_sig_native)

    if np.max(np.abs(sma_sig_native - sma_round)) > eps_sig:
        raise ValueError("Precomputed SMA signals are not integer-like {-1,0,1} values.")
    if np.max(np.abs(ema_sig_native - ema_round)) > eps_sig:
        raise ValueError("Precomputed EMA signals are not integer-like {-1,0,1} values.")

    if not np.all((sma_round >= -1.0) & (sma_round <= 1.0)):
        raise ValueError("Precomputed SMA signals are outside {-1,0,1} range.")
    if not np.all((ema_round >= -1.0) & (ema_round <= 1.0)):
        raise ValueError("Precomputed EMA signals are outside {-1,0,1} range.")

    sma_sig_full = np.ascontiguousarray(sma_round.T.astype(np.int8, copy=False))
    ema_sig_full = np.ascontiguousarray(ema_round.T.astype(np.int8, copy=False))

    signal_open_time_ms = open_time_ms_raw.astype(np.int64, copy=False)
    signal_close_time_ms = close_time_ms_raw.astype(np.int64, copy=False)
    close_signal = arr[:, close_idx].astype(np.float64, copy=False)

    print("  signal source: precomputed columns from prices_and_signals_5m.npy")
    print("  sma contiguous block:", sma_is_block, "ema contiguous block:", ema_is_block)
else:
    if not signal_cols_available:
        raise ValueError(
            "No SMA/EMA signal columns found; recomputation fallback cannot derive window specs."
        )

    # Warmup compile for MA kernels (only on recompute fallback path)
    _ = compute_sma_signal_matrix(np.array([1.0, 2.0, 3.0], dtype=np.float64), np.array([2], dtype=np.int32))
    _ = compute_ema_signal_matrix(np.array([1.0, 2.0, 3.0], dtype=np.float64), np.array([2], dtype=np.int32))

    open_px_raw = arr[:, open_idx].astype(np.float64, copy=False)
    high_px_raw = arr[:, high_idx].astype(np.float64, copy=False)
    low_px_raw = arr[:, low_idx].astype(np.float64, copy=False)
    close_px_raw = arr[:, close_idx].astype(np.float64, copy=False)

    if volume_idx >= 0:
        volume_raw = arr[:, volume_idx].astype(np.float64, copy=False)
    else:
        volume_raw = np.ones(arr.shape[0], dtype=np.float64)

    if is_native_1h_exec:
        ohlc_1h = {
            "open_time_ms": open_time_ms_raw,
            "close_time_ms": close_time_ms_raw,
            "open": open_px_raw,
            "high": high_px_raw,
            "low": low_px_raw,
            "close": close_px_raw,
            "volume": volume_raw,
        }
    else:
        ohlc_1h = aggregate_5m_to_1h(
            open_time_ms=open_time_ms_raw,
            open_px=open_px_raw,
            high_px=high_px_raw,
            low_px=low_px_raw,
            close_px=close_px_raw,
            volume=volume_raw,
        )

    source_arrays_1h = build_source_arrays_from_ohlc(ohlc_1h)

    sma_sources, sma_windows = parse_signal_column_specs(sma_cols)
    ema_sources, ema_windows = parse_signal_column_specs(ema_cols)

    sma_sig_full = build_signal_matrix_for_specs(
        signal_cols=sma_cols,
        sources=sma_sources,
        windows=sma_windows,
        source_arrays=source_arrays_1h,
        kind="sma",
    )
    ema_sig_full = build_signal_matrix_for_specs(
        signal_cols=ema_cols,
        sources=ema_sources,
        windows=ema_windows,
        source_arrays=source_arrays_1h,
        kind="ema",
    )

    signal_open_time_ms = ohlc_1h["open_time_ms"].astype(np.int64, copy=False)
    signal_close_time_ms = ohlc_1h["close_time_ms"].astype(np.int64, copy=False)
    close_signal = ohlc_1h["close"].astype(np.float64, copy=False)
    print("  signal source: recomputed from OHLC fallback path")

(
    detected_signal_dt_ms,
    signal_dt_min_ms,
    signal_dt_max_ms,
    signal_non_pos_deltas,
) = positive_dt_stats_ms(signal_open_time_ms)
if signal_non_pos_deltas > 0:
    raise ValueError(
        "Signal open_time must be strictly increasing. "
        f"Found {signal_non_pos_deltas} non-positive deltas."
    )
effective_signal_tf = timeframe_label_from_dt_ms(int(detected_signal_dt_ms))

n_sig_trade = int(sma_sig_full.shape[1])
if n_sig_trade < 2:
    raise ValueError("Not enough signal bars for strategy evaluation.")
if close_signal.shape[0] != n_sig_trade:
    raise ValueError(
        f"Signal close length mismatch: close={close_signal.shape[0]} signal={n_sig_trade}"
    )

ret_1h = (close_signal[1:] / close_signal[:-1]) - 1.0
ret_1h = ret_1h.astype(np.float32, copy=False)
n_int = int(ret_1h.shape[0])

# Use only bars aligned with ret for E/A/B scoring.
sma_eval_T = np.ascontiguousarray(sma_sig_full[:, :n_int])
ema_eval_T = np.ascontiguousarray(ema_sig_full[:, :n_int])

print("  detected_signal_dt_ms:", int(detected_signal_dt_ms))
print("  detected_signal_tf:", effective_signal_tf)
print("  signal_dt_min_ms:", int(signal_dt_min_ms), "signal_dt_max_ms:", int(signal_dt_max_ms))
if int(signal_dt_min_ms) != int(signal_dt_max_ms):
    print("  warning: signal dt is not constant; median positive delta is used")
print("  signal bars:", n_sig_trade, "(ret intervals:", n_int, ")")
print("  sma_eval_T:", sma_eval_T.shape, sma_eval_T.dtype)
print("  ema_eval_T:", ema_eval_T.shape, ema_eval_T.dtype)
print("  execution bars:", T_exec)


# -------------------------
# 6) E) Preselect SMA/EMA
# -------------------------
NEG_INF = np.float32(-1e30)

sma_nz = (sma_eval_T != 0).sum(axis=1).astype(np.int32)
ema_nz = (ema_eval_T != 0).sum(axis=1).astype(np.int32)

sma_ok = sma_nz >= min_nonzero_single
ema_ok = ema_nz >= min_nonzero_single

print("\nE filter:")
print("  SMA ok:", int(sma_ok.sum()), "/", sma_eval_T.shape[0])
print("  EMA ok:", int(ema_ok.sum()), "/", ema_eval_T.shape[0])

sma_score = single_score_chunked(sma_eval_T, ret_1h, chunk=time_chunk)
ema_score = single_score_chunked(ema_eval_T, ret_1h, chunk=time_chunk)

sma_score_adj = sma_score - (fee_rate * sma_nz.astype(np.float32))
ema_score_adj = ema_score - (fee_rate * ema_nz.astype(np.float32))

sma_score_adj = np.where(sma_ok, sma_score_adj, NEG_INF)
ema_score_adj = np.where(ema_ok, ema_score_adj, NEG_INF)

sma_keep = np.sort(topk_fraction_idx(sma_score_adj, top_frac_side).astype(np.int32))
ema_keep = np.sort(topk_fraction_idx(ema_score_adj, top_frac_side).astype(np.int32))

# Evaluation signals (for E/A/B/D)
sma_eval_T = np.ascontiguousarray(sma_eval_T[sma_keep])
ema_eval_T = np.ascontiguousarray(ema_eval_T[ema_keep])

# Trade signals (full signal bars, incl. last bar)
sma_trade_T = np.ascontiguousarray(sma_sig_full[sma_keep])
ema_trade_T = np.ascontiguousarray(ema_sig_full[ema_keep])

sma_cols_keep = np.array(sma_cols, dtype=object)[sma_keep]
ema_cols_keep = np.array(ema_cols, dtype=object)[ema_keep]

print("After E:")
print("  sma_eval_T:", sma_eval_T.shape)
print("  ema_eval_T:", ema_eval_T.shape)
print("  candidate pairs:", sma_eval_T.shape[0] * ema_eval_T.shape[0])


# -------------------------
# 7) A/B/D) Pairwise precompute via chunked GEMM (on 1h confirmations)
# -------------------------
n_sma, _ = sma_eval_T.shape
n_ema, _ = ema_eval_T.shape
n_pairs = n_sma * n_ema

n_long = np.zeros((n_sma, n_ema), dtype=np.float32)
n_short = np.zeros((n_sma, n_ema), dtype=np.float32)
proxy_score = np.zeros((n_sma, n_ema), dtype=np.float32)

for t0 in tqdm(range(0, n_int, time_chunk), desc="A/B precompute (1h chunked GEMM)"):
    t1 = min(t0 + time_chunk, n_int)
    r = ret_1h[t0:t1]

    sma_chunk = sma_eval_T[:, t0:t1]
    ema_chunk = ema_eval_T[:, t0:t1]

    l_sma = (sma_chunk == 1).astype(np.float32)
    l_ema = (ema_chunk == 1).astype(np.float32)
    n_long += l_sma @ l_ema.T
    proxy_score += l_sma @ (l_ema * r[None, :]).T

    s_sma = (sma_chunk == -1).astype(np.float32)
    s_ema = (ema_chunk == -1).astype(np.float32)
    n_short += s_sma @ s_ema.T
    proxy_score -= s_sma @ (s_ema * r[None, :]).T

n_confirm = n_long + n_short
valid = n_confirm >= float(min_confirm)
valid_cnt = int(valid.sum())

print("\nA/B/D:")
print("  valid pairs:", valid_cnt, "/", n_pairs)

if valid_cnt <= 0:
    raise ValueError("No valid strategies after min_confirm filter on 1h confirmations.")

fee_penalty = np.float32(1.5 * fee_rate) * n_confirm
proxy_adj = proxy_score - fee_penalty
proxy_adj_masked = np.where(valid, proxy_adj, NEG_INF)

k_keep = max(1, int(math.ceil(valid_cnt * top_frac_pairs)))
flat = proxy_adj_masked.ravel()

if k_keep >= flat.size:
    top_idx_flat = np.arange(flat.size, dtype=np.int64)
else:
    top_idx_flat = np.argpartition(flat, flat.size - k_keep)[flat.size - k_keep:]

top_idx_flat = top_idx_flat[flat[top_idx_flat] > NEG_INF / 2]
if top_idx_flat.size == 0:
    top_idx_flat = np.array([int(np.argmax(flat))], dtype=np.int64)

top_si = (top_idx_flat // n_ema).astype(np.int32)
top_ej = (top_idx_flat % n_ema).astype(np.int32)

K = int(top_si.size)
print("  selected strategies:", K)


# -------------------------
# Mapping 1h -> execution indices
# -------------------------
exec_open_ms = open_time_ms_raw
exec_open = arr[:, open_idx].astype(np.float32, copy=False)
exec_close = arr[:, close_idx].astype(np.float32, copy=False)

log_fee_two_sides = float(math.log(fee_two_sides))
NEG_LARGE = -1e30
log_exec_open = np.zeros(T_exec, dtype=np.float64)
_pos_exec_open = exec_open > np.float32(0.0)
if np.any(_pos_exec_open):
    log_exec_open[_pos_exec_open] = np.log(exec_open[_pos_exec_open].astype(np.float64, copy=False))
if T_exec > 0 and float(exec_close[T_exec - 1]) > 0.0:
    log_last_close = float(math.log(float(exec_close[T_exec - 1])))
else:
    log_last_close = 0.0

if is_native_1h_exec:
    # When execution is effectively 1h, entry is strictly next bar in same index space.
    sig_entry_exec_idx = np.arange(n_sig_trade, dtype=np.int32) + np.int32(1)
    sig_entry_exec_idx = np.where(sig_entry_exec_idx >= T_exec, T_exec, sig_entry_exec_idx).astype(np.int32)
else:
    sig_close_ms = signal_close_time_ms
    entry_time_ms = sig_close_ms + np.int64(1)
    sig_entry_exec_idx = np.searchsorted(exec_open_ms, entry_time_ms, side="left").astype(np.int32)
    sig_entry_exec_idx = np.where(sig_entry_exec_idx >= T_exec, T_exec, sig_entry_exec_idx).astype(np.int32)

print("\nMapping 1h -> execution indices:")
print("  sig bars:", sig_entry_exec_idx.shape[0])
print("  first entries:", sig_entry_exec_idx[:5].tolist())
print("  last entries:", sig_entry_exec_idx[-5:].tolist())


# -------------------------
# Trade list construction
# -------------------------
@njit_cached()
def build_trade_list_for_pair(
    sma_sig_row: np.ndarray,
    ema_sig_row: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_entry_exec_idx: np.ndarray,
    out_dir: np.ndarray,
    out_sig_exit_exec_idx: np.ndarray,
) -> np.int32:
    """Build compact trade list from 1h confirmations mapped to execution entries.

    Parameters:
        sma_sig_row: Int8 SMA signal row for one strategy.
        ema_sig_row: Int8 EMA signal row for one strategy.
        sig_entry_exec_idx: 1h-bar to execution-entry mapping.
        T_exec: Number of execution bars.
        out_entry_exec_idx: Preallocated int32 output for trade entries.
        out_dir: Preallocated int8 output for trade directions (+1/-1).
        out_sig_exit_exec_idx: Preallocated int32 output for signal exits.

    Returns:
        Number of trades written into output arrays.
        Returns `-1` when output buffers are insufficient.

    Rules:
        - Confirmation long when SMA+EMA == 2, short when == -2.
        - Repeated confirmations in same direction while already in position are ignored.
        - A trade closes by signal at the next opposite confirmation's entry index.
        - Final open trade gets `t_sig = T_exec` (INF sentinel).

    Assumptions:
        Caller allocates output buffers large enough for all compressed trades.

    Errors:
        Returns `-1` when buffer capacity is exceeded.

    Side effects:
        Writes entries, directions, and signal exits into `out_*` arrays.
    """
    n_sig = sma_sig_row.shape[0]

    n_trades = np.int32(0)
    current_dir = np.int8(0)
    current_entry = np.int32(0)

    for t in range(n_sig):
        comb = np.int16(sma_sig_row[t]) + np.int16(ema_sig_row[t])

        dirn = np.int8(0)
        if comb == 2:
            dirn = np.int8(1)
        elif comb == -2:
            dirn = np.int8(-1)
        else:
            continue

        entry_exec = sig_entry_exec_idx[t]
        if entry_exec >= T_exec:
            break

        if current_dir == 0:
            current_dir = dirn
            current_entry = np.int32(entry_exec)
            continue

        if dirn == current_dir:
            continue

        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)

        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = np.int32(entry_exec)
        n_trades += 1

        current_dir = dirn
        current_entry = np.int32(entry_exec)

    if current_dir != 0:
        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)

        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = T_exec
        n_trades += 1

    return n_trades


@njit_cached(parallel=True)
def count_trades_for_pairs(
    pair_sma_idx: np.ndarray,
    pair_ema_idx: np.ndarray,
    sma_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_trade_counts: np.ndarray,
) -> None:
    """Count trades per selected strategy after same-direction compression.

    Parameters:
        pair_sma_idx: Selected SMA row indices.
        pair_ema_idx: Selected EMA row indices.
        sma_trade_T: Full 1h SMA signal matrix (selected rows).
        ema_trade_T: Full 1h EMA signal matrix (selected rows).
        sig_entry_exec_idx: Mapping from 1h close to execution entry index.
        T_exec: Number of execution bars.
        out_trade_counts: Output int32 trade counts per strategy.

    Side effects:
        Writes counts into `out_trade_counts`.
    """
    K_ = pair_sma_idx.shape[0]
    n_sig = sma_trade_T.shape[1]

    for k in nb.prange(K_):
        si = pair_sma_idx[k]
        ej = pair_ema_idx[k]

        tmp_entry = np.empty(n_sig, dtype=np.int32)
        tmp_dir = np.empty(n_sig, dtype=np.int8)
        tmp_exit = np.empty(n_sig, dtype=np.int32)

        out_trade_counts[k] = build_trade_list_for_pair(
            sma_trade_T[si],
            ema_trade_T[ej],
            sig_entry_exec_idx,
            T_exec,
            tmp_entry,
            tmp_dir,
            tmp_exit,
        )


@njit_cached(inline="always")
def evaluate_trade_factor(
    dirn: np.int8,
    entry_exec: np.int32,
    sig_exit_exec: np.int32,
    tp_i: np.int32,
    sl_i: np.int32,
    exec_open: np.ndarray,
    last_close: float,
    T_exec: np.int32,
    close_on_end: np.int8,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
) -> tuple[float, np.int32, np.int8]:
    """Resolve one trade exit and return gross factor before fees.

    Parameters:
        dirn: Trade direction (+1 long, -1 short).
        entry_exec: Execution-bar entry index.
        sig_exit_exec: Next opposite-signal entry index, or `T_exec`.
        tp_i: TP grid index.
        sl_i: SL grid index.
        exec_open: Execution-bar open prices.
        last_close: Final execution-bar close for optional end-of-test close.
        T_exec: Number of execution bars.
        close_on_end: Whether to force close final open trade at end.
        hit_*: Hit-time lookup tables.
        *_eq: Price factors for TP/SL level fills.

    Returns:
        Tuple `(gross_factor, exit_exec_idx, closed_flag)`.

    Rules:
        - TP/SL lookup starts from `entry_exec + 1` to avoid entry-bar lookahead.
        - Exit event compares TP/SL vs signal-exit with signal priority on equal bar.
        - SL wins TP/SL ties when both happen before signal-exit.
        - Signal exits use execution-bar open at `t_sig`.
        - TP/SL exits use exact level factors.
        - Short signal/end exits use x1 USDT ROI: `pf = max(0, 2 - exit/entry)`.
    """
    entry_open = float(exec_open[entry_exec])
    if entry_open <= 0.0:
        return 1.0, entry_exec, np.int8(0)

    lookup_exec = np.int32(entry_exec + 1)

    ttp = T_exec
    tsl = T_exec
    tp_pf = 1.0
    sl_pf = 1.0

    if lookup_exec < T_exec:
        if dirn == 1:
            ttp = np.int32(hit_long_tp[tp_i, lookup_exec])
            tsl = np.int32(hit_long_sl[sl_i, lookup_exec])
            tp_pf = float(long_tp_eq[tp_i])
            sl_pf = float(long_sl_eq[sl_i])
        else:
            ttp = np.int32(hit_short_tp[tp_i, lookup_exec])
            tsl = np.int32(hit_short_sl[sl_i, lookup_exec])
            tp_pf = float(short_tp_eq[tp_i])
            sl_pf = float(short_sl_eq[sl_i])

    # TP/SL candidate with conservative tie (SL wins TP/SL tie).
    if tsl <= ttp:
        tp_sl_exec = tsl
        tp_sl_pf = sl_pf
    else:
        tp_sl_exec = ttp
        tp_sl_pf = tp_pf

    # Signal-exit at bar open should win on equal execution bar.
    if sig_exit_exec < T_exec and sig_exit_exec <= tp_sl_exec:
        exit_open = float(exec_open[sig_exit_exec])
        if exit_open > 0.0:
            if dirn == 1:
                pf = exit_open / entry_open
            else:
                ratio = exit_open / entry_open
                pf = 2.0 - ratio
                if pf <= 0.0:
                    pf = 0.0
        else:
            pf = 1.0
        return pf, sig_exit_exec, np.int8(1)

    if tp_sl_exec < T_exec:
        return tp_sl_pf, tp_sl_exec, np.int8(1)

    if close_on_end == 1 and T_exec > 0:
        if last_close > 0.0:
            if dirn == 1:
                pf = last_close / entry_open
            else:
                ratio = last_close / entry_open
                pf = 2.0 - ratio
                if pf <= 0.0:
                    pf = 0.0
        else:
            pf = 1.0
        return pf, np.int32(T_exec - 1), np.int8(1)

    return 1.0, entry_exec, np.int8(0)


# -------------------------
# TP/SL grid evaluation (best tp/sl per strategy)
# -------------------------
@njit_cached(inline="always")
def add_row_range(
    row_diff: np.ndarray,
    row_i: np.int32,
    col_start: np.int32,
    col_stop: np.int32,
    value: float,
) -> None:
    """Apply row segment addition using a 1D difference array.

    Parameters:
        row_diff: Row-wise diff buffer with shape `(n_tp, n_sl + 1)`.
        row_i: Target row index.
        col_start: Inclusive start column.
        col_stop: Exclusive stop column.
        value: Additive contribution in log-equity space.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Side effects:
        Mutates `row_diff` in place.
    """
    if col_start < col_stop:
        row_diff[row_i, col_start] += value
        row_diff[row_i, col_stop] -= value


@njit_cached(inline="always")
def add_col_range(
    col_diff: np.ndarray,
    row_start: np.int32,
    row_stop: np.int32,
    col_j: np.int32,
    value: float,
) -> None:
    """Apply column segment addition using a 1D difference array.

    Parameters:
        col_diff: Column-wise diff buffer with shape `(n_tp + 1, n_sl)`.
        row_start: Inclusive start row.
        row_stop: Exclusive stop row.
        col_j: Target column index.
        value: Additive contribution in log-equity space.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Side effects:
        Mutates `col_diff` in place.
    """
    if row_start < row_stop:
        col_diff[row_start, col_j] += value
        col_diff[row_stop, col_j] -= value


@njit_cached(inline="always")
def add_rect(
    rect_diff: np.ndarray,
    row_start: np.int32,
    col_start: np.int32,
    row_stop: np.int32,
    col_stop: np.int32,
    value: float,
) -> None:
    """Apply rectangle addition using a 2D difference array.

    Parameters:
        rect_diff: 2D diff buffer with shape `(n_tp + 1, n_sl + 1)`.
        row_start: Inclusive start row.
        col_start: Inclusive start column.
        row_stop: Exclusive stop row.
        col_stop: Exclusive stop column.
        value: Additive contribution in log-equity space.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Side effects:
        Mutates `rect_diff` in place.
    """
    if row_start < row_stop and col_start < col_stop:
        rect_diff[row_start, col_start] += value
        rect_diff[row_stop, col_start] -= value
        rect_diff[row_start, col_stop] -= value
        rect_diff[row_stop, col_stop] += value


@njit_cached(inline="always")
def is_non_decreasing_hit_levels(
    hit_table: np.ndarray,
    start_exec: np.int32,
    n_levels: np.int32,
) -> np.int8:
    """Check monotonic hit-time ordering across level axis at one start index.

    Parameters:
        hit_table: Hit-time table shaped `(n_levels, T_exec)`.
        start_exec: Execution index for lookup.
        n_levels: Number of levels to scan.

    Returns:
        `1` if values are non-decreasing, else `0`.

    Assumptions:
        `start_exec` is a valid column index.

    Side effects:
        None.
    """
    if n_levels <= 1:
        return np.int8(1)

    prev = np.int32(hit_table[0, start_exec])
    for i in range(1, n_levels):
        cur = np.int32(hit_table[i, start_exec])
        if cur < prev:
            return np.int8(0)
        prev = cur

    return np.int8(1)


@njit_cached(inline="always")
def lower_bound_ge_hit(
    hit_table: np.ndarray,
    start_exec: np.int32,
    n_levels: np.int32,
    target: np.int32,
) -> np.int32:
    """Find first level whose hit-time is greater than or equal to target.

    Parameters:
        hit_table: Monotone hit-time table `(n_levels, T_exec)`.
        start_exec: Execution index for lookup.
        n_levels: Number of levels in the table axis.
        target: Target execution index.

    Returns:
        First level index `i` with `hit_table[i, start_exec] >= target`,
        or `n_levels` if no such level exists.

    Assumptions:
        Hit times are non-decreasing across levels for this start index.

    Side effects:
        None.
    """
    lo = np.int32(0)
    hi = np.int32(n_levels)

    while lo < hi:
        mid = np.int32((lo + hi) // 2)
        if np.int32(hit_table[mid, start_exec]) >= target:
            hi = mid
        else:
            lo = np.int32(mid + 1)

    return lo


@njit_cached(inline="always")
def first_equal_hit(
    hit_table: np.ndarray,
    start_exec: np.int32,
    n_levels: np.int32,
    target: np.int32,
) -> np.int32:
    """Find first level equal to target in a monotone hit-time axis.

    Parameters:
        hit_table: Monotone hit-time table `(n_levels, T_exec)`.
        start_exec: Execution index for lookup.
        n_levels: Number of levels in the table axis.
        target: Target value to match.

    Returns:
        First level index with exact match, or `n_levels` if no match exists.

    Assumptions:
        Hit times are non-decreasing across levels for this start index.

    Side effects:
        None.
    """
    idx = lower_bound_ge_hit(hit_table, start_exec, n_levels, target)
    if idx < n_levels and np.int32(hit_table[idx, start_exec]) == target:
        return idx
    return np.int32(n_levels)


@njit_cached(inline="always")
def add_trade_full_grid_fallback(
    dirn: np.int8,
    entry_exec: np.int32,
    sig_exit_exec: np.int32,
    exec_open: np.ndarray,
    last_close: float,
    T_exec: np.int32,
    close_on_end: np.int8,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    fee_two_sides: float,
    rect_diff: np.ndarray,
) -> None:
    """Fallback one-trade update by explicit TP/SL cell replay.

    Parameters:
        dirn: Trade direction (+1/-1).
        entry_exec: Entry execution index.
        sig_exit_exec: Signal exit execution index or `T_exec`.
        exec_open: Execution open prices.
        last_close: Final execution close for optional end-close logic.
        T_exec: Number of execution bars.
        close_on_end: Whether to force-close open trade at test end.
        hit_*: Hit-time lookup tables.
        *_eq: Exact TP/SL gross factors.
        fee_two_sides: Two-sided fee factor `(1-fee_rate)^2`.
        rect_diff: 2D diff accumulator for rectangle updates.

    Returns:
        None.

    Assumptions:
        Used only when monotonic decomposition prerequisites are violated.

    Side effects:
        Adds per-cell log contributions directly into `rect_diff`.
    """
    n_tp_ = hit_long_tp.shape[0]
    n_sl_ = hit_long_sl.shape[0]

    for tp_i in range(n_tp_):
        for sl_i in range(n_sl_):
            pf, _, closed = evaluate_trade_factor(
                dirn,
                entry_exec,
                sig_exit_exec,
                np.int32(tp_i),
                np.int32(sl_i),
                exec_open,
                last_close,
                T_exec,
                close_on_end,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                long_tp_eq,
                long_sl_eq,
                short_tp_eq,
                short_sl_eq,
            )

            if closed == 1:
                add_rect(
                    rect_diff,
                    np.int32(tp_i),
                    np.int32(sl_i),
                    np.int32(tp_i + 1),
                    np.int32(sl_i + 1),
                    math.log(fee_two_sides * pf),
                )


# -------------------------
# TP/SL grid evaluation (best tp/sl per strategy)
# -------------------------
@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_trade_list_slow(
    pair_sma_idx: np.ndarray,
    pair_ema_idx: np.ndarray,
    sma_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open: np.ndarray,
    exec_close: np.ndarray,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    fee_rate: float,
    close_on_end: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_counts: np.ndarray,
) -> None:
    """Reference TP/SL grid search by explicit cell replay.

    Parameters:
        pair_sma_idx: Selected SMA indices.
        pair_ema_idx: Selected EMA indices.
        sma_trade_T: 1h SMA signal matrix for selected rows.
        ema_trade_T: 1h EMA signal matrix for selected rows.
        sig_entry_exec_idx: 1h-to-execution entry mapping.
        exec_open: Execution-bar open prices.
        exec_close: Execution-bar close prices.
        T_exec: Number of execution bars.
        hit_*: TP/SL hit-time tables.
        *_eq: TP/SL gross level factors.
        fee_rate: Fee per side.
        close_on_end: Whether to close final open trade at end.
        out_best_*: Output arrays for best indices and return.
        out_trade_counts: Output trade counts after compression.

    Returns:
        None.

    Assumptions:
        Trade-list arrays are rebuilt per strategy and fit preallocated buffers.

    Side effects:
        Writes best TP/SL cell, return, and trade counts per strategy.

    Complexity:
        `O(strategy_count * trade_count * n_tp * n_sl)`.
    """
    K_ = pair_sma_idx.shape[0]
    n_tp_ = hit_long_tp.shape[0]
    n_sl_ = hit_long_sl.shape[0]
    n_sig = sma_trade_T.shape[1]

    fee_two_sides_local = (1.0 - fee_rate) * (1.0 - fee_rate)
    last_close = float(exec_close[T_exec - 1])

    for k in nb.prange(K_):
        si = pair_sma_idx[k]
        ej = pair_ema_idx[k]

        entry_arr = np.empty(n_sig, dtype=np.int32)
        dir_arr = np.empty(n_sig, dtype=np.int8)
        sig_exit_arr = np.empty(n_sig, dtype=np.int32)

        n_trades = build_trade_list_for_pair(
            sma_trade_T[si],
            ema_trade_T[ej],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )
        out_trade_counts[k] = n_trades

        if n_trades <= 0:
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        best_eq = -1.0
        best_tp = np.int32(0)
        best_sl = np.int32(0)

        for tp_i in range(n_tp_):
            for sl_i in range(n_sl_):
                eq = 1.0

                for tr in range(n_trades):
                    pf, _, closed = evaluate_trade_factor(
                        dir_arr[tr],
                        entry_arr[tr],
                        sig_exit_arr[tr],
                        np.int32(tp_i),
                        np.int32(sl_i),
                        exec_open,
                        last_close,
                        T_exec,
                        close_on_end,
                        hit_long_tp,
                        hit_long_sl,
                        hit_short_tp,
                        hit_short_sl,
                        long_tp_eq,
                        long_sl_eq,
                        short_tp_eq,
                        short_sl_eq,
                    )

                    if closed == 1:
                        eq *= fee_two_sides_local * pf

                if eq > best_eq:
                    best_eq = eq
                    best_tp = np.int32(tp_i)
                    best_sl = np.int32(sl_i)

        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl
        out_best_ret[k] = np.float32(best_eq - 1.0)


@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_trade_list_fast_monotone(
    pair_sma_idx: np.ndarray,
    pair_ema_idx: np.ndarray,
    sma_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open: np.ndarray,
    exec_close: np.ndarray,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    trade_counts: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    log_exec_open: np.ndarray,
    log_last_close: float,
    fee_two_sides: float,
    close_on_end: np.int8,
    debug_checks: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_counts: np.ndarray,
) -> None:
    """Find best TP/SL per strategy with monotone hit-time decomposition.

    Parameters:
        pair_sma_idx: Selected SMA row indices.
        pair_ema_idx: Selected EMA row indices.
        sma_trade_T: Full SMA signal matrix for selected rows.
        ema_trade_T: Full EMA signal matrix for selected rows.
        sig_entry_exec_idx: Signal-bar to execution-entry mapping.
        exec_open: Execution open prices.
        exec_close: Execution close prices.
        T_exec: Number of execution bars.
        hit_long_tp: Long TP hit-time table.
        hit_long_sl: Long SL hit-time table.
        hit_short_tp: Short TP hit-time table.
        hit_short_sl: Short SL hit-time table.
        long_tp_eq: Long TP gross factors.
        long_sl_eq: Long SL gross factors.
        short_tp_eq: Short TP gross factors.
        short_sl_eq: Short SL gross factors.
        trade_counts: Precomputed trades per strategy for compact allocations.
        log_fac_tp_long: Log factors for long TP exits including fees.
        log_fac_sl_long: Log factors for long SL exits including fees.
        log_fac_tp_short: Log factors for short TP exits including fees.
        log_fac_sl_short: Log factors for short SL exits including fees.
        log_fee_two_sides: Log of two-sided fee factor.
        log_exec_open: Log-open vector with zeros where open <= 0.
        log_last_close: Log of final close if positive, otherwise zero.
        fee_two_sides: Two-sided fee factor.
        close_on_end: Close-open-trade-at-end switch.
        debug_checks: Enables extra sentinel checks when set to `1`.
        out_best_tp_idx: Output best TP index per strategy.
        out_best_sl_idx: Output best SL index per strategy.
        out_best_ret: Output best return per strategy.
        out_trade_counts: Output trade count per strategy.

    Returns:
        None.

    Assumptions:
        Hit-time tables are monotone non-decreasing across level axis.

    Errors:
        None directly; debug mode writes sentinel counts `-1`/`-2` on issues.

    Side effects:
        Writes best-cell outputs and trade counts into provided arrays.
    """
    K_ = pair_sma_idx.shape[0]
    n_tp_ = hit_long_tp.shape[0]
    n_sl_ = hit_long_sl.shape[0]

    last_close = 0.0
    if T_exec > 0:
        last_close = float(exec_close[T_exec - 1])

    for k in nb.prange(K_):
        si = pair_sma_idx[k]
        ej = pair_ema_idx[k]

        alloc_n = np.int32(trade_counts[k])
        if alloc_n <= 0:
            out_trade_counts[k] = np.int32(0)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        entry_arr = np.empty(alloc_n, dtype=np.int32)
        dir_arr = np.empty(alloc_n, dtype=np.int8)
        sig_exit_arr = np.empty(alloc_n, dtype=np.int32)

        n_trades = build_trade_list_for_pair(
            sma_trade_T[si],
            ema_trade_T[ej],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )

        if n_trades < 0:
            out_trade_counts[k] = n_trades
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        if debug_checks == 1 and n_trades > alloc_n:
            out_trade_counts[k] = np.int32(-2)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        if n_trades <= 0:
            out_trade_counts[k] = np.int32(0)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        out_trade_counts[k] = n_trades

        row_diff = np.zeros((n_tp_, n_sl_ + 1), dtype=np.float64)
        col_diff = np.zeros((n_tp_ + 1, n_sl_), dtype=np.float64)
        rect_diff = np.zeros((n_tp_ + 1, n_sl_ + 1), dtype=np.float64)

        for tr in range(n_trades):
            dirn = dir_arr[tr]
            entry_exec = entry_arr[tr]
            sig_exit_exec = sig_exit_arr[tr]

            entry_open = float(exec_open[entry_exec])
            if entry_open <= 0.0:
                continue
            entry_log = float(log_exec_open[entry_exec])

            start = np.int32(entry_exec + 1)

            if dirn == 1:
                hit_tp = hit_long_tp
                hit_sl = hit_long_sl
                log_tp_arr = log_fac_tp_long
                log_sl_arr = log_fac_sl_long
            else:
                hit_tp = hit_short_tp
                hit_sl = hit_short_sl
                log_tp_arr = log_fac_tp_short
                log_sl_arr = log_fac_sl_short

            if start >= T_exec:
                if sig_exit_exec < T_exec:
                    exit_open = float(exec_open[sig_exit_exec])
                    contrib = log_fee_two_sides
                    if exit_open > 0.0:
                        if dirn == 1:
                            exit_log = float(log_exec_open[sig_exit_exec])
                            contrib += exit_log - entry_log
                        else:
                            ratio = exit_open / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_), np.int32(n_sl_), contrib)
                elif close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close > 0.0:
                        if dirn == 1:
                            contrib += log_last_close - entry_log
                        else:
                            ratio = last_close / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_), np.int32(n_sl_), contrib)
                continue

            if sig_exit_exec < T_exec:
                t_sig = np.int32(sig_exit_exec)

                i_sig = lower_bound_ge_hit(hit_tp, start, np.int32(n_tp_), t_sig)
                j_sig = lower_bound_ge_hit(hit_sl, start, np.int32(n_sl_), t_sig)

                exit_open = float(exec_open[sig_exit_exec])
                contrib = log_fee_two_sides
                if exit_open > 0.0:
                    if dirn == 1:
                        exit_log = float(log_exec_open[sig_exit_exec])
                        contrib += exit_log - entry_log
                    else:
                        ratio = exit_open / entry_open
                        if ratio >= 2.0:
                            contrib = NEG_LARGE
                        else:
                            contrib += math.log(2.0 - ratio)

                add_rect(
                    rect_diff,
                    i_sig,
                    j_sig,
                    np.int32(n_tp_),
                    np.int32(n_sl_),
                    contrib,
                )

                j_ptr = np.int32(0)
                for i in range(i_sig):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < j_sig and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(
                        row_diff,
                        np.int32(i),
                        j_ptr,
                        np.int32(n_sl_),
                        float(log_tp_arr[i]),
                    )

                i_ptr = np.int32(0)
                for j in range(j_sig):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_sig and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(
                        col_diff,
                        i_ptr,
                        np.int32(n_tp_),
                        np.int32(j),
                        float(log_sl_arr[j]),
                    )

            else:
                i_never = first_equal_hit(hit_tp, start, np.int32(n_tp_), T_exec)
                j_never = first_equal_hit(hit_sl, start, np.int32(n_sl_), T_exec)

                if close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close > 0.0:
                        if dirn == 1:
                            contrib += log_last_close - entry_log
                        else:
                            ratio = last_close / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)

                    add_rect(
                        rect_diff,
                        i_never,
                        j_never,
                        np.int32(n_tp_),
                        np.int32(n_sl_),
                        contrib,
                    )

                j_ptr = np.int32(0)
                for i in range(i_never):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < n_sl_ and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(
                        row_diff,
                        np.int32(i),
                        j_ptr,
                        np.int32(n_sl_),
                        float(log_tp_arr[i]),
                    )

                i_ptr = np.int32(0)
                for j in range(j_never):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_never and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(
                        col_diff,
                        i_ptr,
                        np.int32(n_tp_),
                        np.int32(j),
                        float(log_sl_arr[j]),
                    )

        for i in range(n_tp_):
            run = 0.0
            for j in range(n_sl_):
                run += row_diff[i, j]
                row_diff[i, j] = run

        for j in range(n_sl_):
            run = 0.0
            for i in range(n_tp_):
                run += col_diff[i, j]
                col_diff[i, j] = run

        for i in range(n_tp_):
            row_run = 0.0
            for j in range(n_sl_):
                row_run += rect_diff[i, j]
                if i == 0:
                    rect_diff[i, j] = row_run
                else:
                    rect_diff[i, j] = row_run + rect_diff[i - 1, j]

        best_log = -1.0e300
        best_tp = np.int32(0)
        best_sl = np.int32(0)

        for tp_i in range(n_tp_):
            for sl_i in range(n_sl_):
                v = row_diff[tp_i, sl_i] + col_diff[tp_i, sl_i] + rect_diff[tp_i, sl_i]
                if v > best_log:
                    best_log = v
                    best_tp = np.int32(tp_i)
                    best_sl = np.int32(sl_i)

        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl

        best_eq = 1.0
        for tr in range(n_trades):
            pf, _, closed = evaluate_trade_factor(
                dir_arr[tr],
                entry_arr[tr],
                sig_exit_arr[tr],
                best_tp,
                best_sl,
                exec_open,
                last_close,
                T_exec,
                close_on_end,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                long_tp_eq,
                long_sl_eq,
                short_tp_eq,
                short_sl_eq,
            )
            if closed == 1:
                best_eq *= fee_two_sides * pf

        out_best_ret[k] = np.float32(best_eq - 1.0)


@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_trade_list_fast_monotone_f32(
    pair_sma_idx: np.ndarray,
    pair_ema_idx: np.ndarray,
    sma_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open: np.ndarray,
    exec_close: np.ndarray,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    trade_counts: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    log_exec_open: np.ndarray,
    log_last_close: float,
    fee_two_sides: float,
    close_on_end: np.int8,
    debug_checks: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_counts: np.ndarray,
) -> None:
    """Find best TP/SL per strategy using float32 diff buffers.

    Parameters:
        Same as `evaluate_best_tp_sl_trade_list_fast_monotone`.

    Returns:
        None.

    Assumptions:
        Hit-time tables are monotone non-decreasing across level axis.

    Errors:
        None directly; debug mode writes sentinel counts `-1`/`-2` on issues.

    Side effects:
        Writes best-cell outputs and trade counts into provided arrays.
    """
    K_ = pair_sma_idx.shape[0]
    n_tp_ = hit_long_tp.shape[0]
    n_sl_ = hit_long_sl.shape[0]

    last_close = 0.0
    if T_exec > 0:
        last_close = float(exec_close[T_exec - 1])

    for k in nb.prange(K_):
        si = pair_sma_idx[k]
        ej = pair_ema_idx[k]

        alloc_n = np.int32(trade_counts[k])
        if alloc_n <= 0:
            out_trade_counts[k] = np.int32(0)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        entry_arr = np.empty(alloc_n, dtype=np.int32)
        dir_arr = np.empty(alloc_n, dtype=np.int8)
        sig_exit_arr = np.empty(alloc_n, dtype=np.int32)

        n_trades = build_trade_list_for_pair(
            sma_trade_T[si],
            ema_trade_T[ej],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )

        if n_trades < 0:
            out_trade_counts[k] = n_trades
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        if debug_checks == 1 and n_trades > alloc_n:
            out_trade_counts[k] = np.int32(-2)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        if n_trades <= 0:
            out_trade_counts[k] = np.int32(0)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        out_trade_counts[k] = n_trades

        row_diff = np.zeros((n_tp_, n_sl_ + 1), dtype=np.float32)
        col_diff = np.zeros((n_tp_ + 1, n_sl_), dtype=np.float32)
        rect_diff = np.zeros((n_tp_ + 1, n_sl_ + 1), dtype=np.float32)

        for tr in range(n_trades):
            dirn = dir_arr[tr]
            entry_exec = entry_arr[tr]
            sig_exit_exec = sig_exit_arr[tr]

            entry_open = float(exec_open[entry_exec])
            if entry_open <= 0.0:
                continue
            entry_log = float(log_exec_open[entry_exec])

            start = np.int32(entry_exec + 1)

            if dirn == 1:
                hit_tp = hit_long_tp
                hit_sl = hit_long_sl
                log_tp_arr = log_fac_tp_long
                log_sl_arr = log_fac_sl_long
            else:
                hit_tp = hit_short_tp
                hit_sl = hit_short_sl
                log_tp_arr = log_fac_tp_short
                log_sl_arr = log_fac_sl_short

            if start >= T_exec:
                if sig_exit_exec < T_exec:
                    exit_open = float(exec_open[sig_exit_exec])
                    contrib = log_fee_two_sides
                    if exit_open > 0.0:
                        if dirn == 1:
                            exit_log = float(log_exec_open[sig_exit_exec])
                            contrib += exit_log - entry_log
                        else:
                            ratio = exit_open / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_), np.int32(n_sl_), np.float32(contrib))
                elif close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close > 0.0:
                        if dirn == 1:
                            contrib += log_last_close - entry_log
                        else:
                            ratio = last_close / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_), np.int32(n_sl_), np.float32(contrib))
                continue

            if sig_exit_exec < T_exec:
                t_sig = np.int32(sig_exit_exec)

                i_sig = lower_bound_ge_hit(hit_tp, start, np.int32(n_tp_), t_sig)
                j_sig = lower_bound_ge_hit(hit_sl, start, np.int32(n_sl_), t_sig)

                exit_open = float(exec_open[sig_exit_exec])
                contrib = log_fee_two_sides
                if exit_open > 0.0:
                    if dirn == 1:
                        exit_log = float(log_exec_open[sig_exit_exec])
                        contrib += exit_log - entry_log
                    else:
                        ratio = exit_open / entry_open
                        if ratio >= 2.0:
                            contrib = NEG_LARGE
                        else:
                            contrib += math.log(2.0 - ratio)

                add_rect(
                    rect_diff,
                    i_sig,
                    j_sig,
                    np.int32(n_tp_),
                    np.int32(n_sl_),
                    np.float32(contrib),
                )

                j_ptr = np.int32(0)
                for i in range(i_sig):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < j_sig and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(
                        row_diff,
                        np.int32(i),
                        j_ptr,
                        np.int32(n_sl_),
                        np.float32(log_tp_arr[i]),
                    )

                i_ptr = np.int32(0)
                for j in range(j_sig):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_sig and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(
                        col_diff,
                        i_ptr,
                        np.int32(n_tp_),
                        np.int32(j),
                        np.float32(log_sl_arr[j]),
                    )

            else:
                i_never = first_equal_hit(hit_tp, start, np.int32(n_tp_), T_exec)
                j_never = first_equal_hit(hit_sl, start, np.int32(n_sl_), T_exec)

                if close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close > 0.0:
                        if dirn == 1:
                            contrib += log_last_close - entry_log
                        else:
                            ratio = last_close / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)

                    add_rect(
                        rect_diff,
                        i_never,
                        j_never,
                        np.int32(n_tp_),
                        np.int32(n_sl_),
                        np.float32(contrib),
                    )

                j_ptr = np.int32(0)
                for i in range(i_never):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < n_sl_ and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(
                        row_diff,
                        np.int32(i),
                        j_ptr,
                        np.int32(n_sl_),
                        np.float32(log_tp_arr[i]),
                    )

                i_ptr = np.int32(0)
                for j in range(j_never):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_never and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(
                        col_diff,
                        i_ptr,
                        np.int32(n_tp_),
                        np.int32(j),
                        np.float32(log_sl_arr[j]),
                    )

        for i in range(n_tp_):
            run = np.float32(0.0)
            for j in range(n_sl_):
                run += row_diff[i, j]
                row_diff[i, j] = run

        for j in range(n_sl_):
            run = np.float32(0.0)
            for i in range(n_tp_):
                run += col_diff[i, j]
                col_diff[i, j] = run

        for i in range(n_tp_):
            row_run = np.float32(0.0)
            for j in range(n_sl_):
                row_run += rect_diff[i, j]
                if i == 0:
                    rect_diff[i, j] = row_run
                else:
                    rect_diff[i, j] = row_run + rect_diff[i - 1, j]

        best_log = np.float32(-3.4028235e38)
        best_tp = np.int32(0)
        best_sl = np.int32(0)

        for tp_i in range(n_tp_):
            for sl_i in range(n_sl_):
                v = row_diff[tp_i, sl_i] + col_diff[tp_i, sl_i] + rect_diff[tp_i, sl_i]
                if v > best_log:
                    best_log = v
                    best_tp = np.int32(tp_i)
                    best_sl = np.int32(sl_i)

        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl

        best_eq = 1.0
        for tr in range(n_trades):
            pf, _, closed = evaluate_trade_factor(
                dir_arr[tr],
                entry_arr[tr],
                sig_exit_arr[tr],
                best_tp,
                best_sl,
                exec_open,
                last_close,
                T_exec,
                close_on_end,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                long_tp_eq,
                long_sl_eq,
                short_tp_eq,
                short_sl_eq,
            )
            if closed == 1:
                best_eq *= fee_two_sides * pf

        out_best_ret[k] = np.float32(best_eq - 1.0)


# Fast path is the default grid-search kernel used below.


# ---- Numba threads
try:
    nb.set_num_threads(os.cpu_count() or 8)
    print("\nNumba threads:", nb.get_num_threads())
except Exception as e:
    print("\nNumba set threads skipped:", e)

# Warmup for trade count
trade_counts_preview = np.empty(K, dtype=np.int32)
w = min(8, K)
count_trades_for_pairs(
    top_si[:w],
    top_ej[:w],
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    np.int32(T_exec),
    trade_counts_preview[:w],
)

trade_counts = np.empty(K, dtype=np.int32)
t_count0 = time.perf_counter()
count_trades_for_pairs(
    top_si,
    top_ej,
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    np.int32(T_exec),
    trade_counts,
)
t_count1 = time.perf_counter()

confirm_counts_selected = n_confirm[top_si, top_ej].astype(np.int64)

print("\nTrade list construction:")
print("  mean confirmations per strategy:", float(confirm_counts_selected.mean()))
print("  mean trades per strategy:", float(trade_counts.mean()))
print("  median confirmations per strategy:", float(np.median(confirm_counts_selected)))
print("  median trades per strategy:", float(np.median(trade_counts)))
print("  trade-count kernel time (s):", round(t_count1 - t_count0, 3))

print("\nSanity & Diagnostics:")
print("  detected_exec_dt_ms:", detected_dt_ms)
print("  detected_signal_dt_ms:", int(detected_signal_dt_ms))
print("  detected_exec_tf:", effective_exec_tf)
print("  detected_signal_tf:", effective_signal_tf)
print("  effective_exec_tf:", effective_exec_tf)
print("  signal bars:", n_sig_trade)
print("  execution bars:", T_exec)
print("  exec_dt_min_ms:", exec_dt_min_ms, "exec_dt_max_ms:", exec_dt_max_ms)
print("  signal_dt_min_ms:", int(signal_dt_min_ms), "signal_dt_max_ms:", int(signal_dt_max_ms))
print("  mean confirmations per strategy:", float(confirm_counts_selected.mean()))
print("  mean trades per strategy:", float(trade_counts.mean()))

best_tp_idx = np.empty(K, dtype=np.int32)
best_sl_idx = np.empty(K, dtype=np.int32)
best_ret = np.empty(K, dtype=np.float32)
grid_trade_counts = np.empty(K, dtype=np.int32)

GRID_SELF_CHECK_ENABLED = int(os.environ.get("BT_GRID_SELF_CHECK", "0")) == 1
GRID_SELF_CHECK_N = int(os.environ.get("BT_GRID_SELF_CHECK_N", "50"))
if GRID_SELF_CHECK_N < 1:
    GRID_SELF_CHECK_N = 1
GRID_BENCHMARK_SLOW_FULL = int(os.environ.get("BT_GRID_BENCH_SLOW", "0")) == 1
GRID_DIFF_F32 = int(os.environ.get("BT_GRID_DIFF_F32", "0")) == 1

grid_fast_kernel = (
    evaluate_best_tp_sl_trade_list_fast_monotone_f32
    if GRID_DIFF_F32
    else evaluate_best_tp_sl_trade_list_fast_monotone
)

print(
    "\nGrid debug toggles:",
    "self_check=",
    GRID_SELF_CHECK_ENABLED,
    "self_check_n=",
    GRID_SELF_CHECK_N,
    "bench_slow_full=",
    GRID_BENCHMARK_SLOW_FULL,
    "diff_f32=",
    GRID_DIFF_F32,
)

# Warmup compile for fast grid kernel
grid_fast_kernel(
    top_si[:w],
    top_ej[:w],
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    exec_open,
    exec_close,
    np.int32(T_exec),
    hit_long_tp,
    hit_long_sl,
    hit_short_tp,
    hit_short_sl,
    long_tp_eq,
    long_sl_eq,
    short_tp_eq,
    short_sl_eq,
    trade_counts[:w],
    log_fac_tp_long,
    log_fac_sl_long,
    log_fac_tp_short,
    log_fac_sl_short,
    float(log_fee_two_sides),
    log_exec_open,
    float(log_last_close),
    float(fee_two_sides),
    np.int8(close_on_end),
    np.int8(1),
    best_tp_idx[:w],
    best_sl_idx[:w],
    best_ret[:w],
    grid_trade_counts[:w],
)

if GRID_SELF_CHECK_ENABLED or GRID_BENCHMARK_SLOW_FULL:
    sw = min(2, w)
    if sw > 0:
        _slow_tp = np.empty(sw, dtype=np.int32)
        _slow_sl = np.empty(sw, dtype=np.int32)
        _slow_ret = np.empty(sw, dtype=np.float32)
        _slow_cnt = np.empty(sw, dtype=np.int32)
        evaluate_best_tp_sl_trade_list_slow(
            top_si[:sw],
            top_ej[:sw],
            sma_trade_T,
            ema_trade_T,
            sig_entry_exec_idx,
            exec_open,
            exec_close,
            np.int32(T_exec),
            hit_long_tp,
            hit_long_sl,
            hit_short_tp,
            hit_short_sl,
            long_tp_eq,
            long_sl_eq,
            short_tp_eq,
            short_sl_eq,
            float(fee_rate),
            np.int8(close_on_end),
            _slow_tp,
            _slow_sl,
            _slow_ret,
            _slow_cnt,
        )

if GRID_SELF_CHECK_ENABLED:
    chk = min(GRID_SELF_CHECK_N, K)
    if chk > 0:
        slow_tp_chk = np.empty(chk, dtype=np.int32)
        slow_sl_chk = np.empty(chk, dtype=np.int32)
        slow_ret_chk = np.empty(chk, dtype=np.float32)
        slow_cnt_chk = np.empty(chk, dtype=np.int32)

        fast_tp_chk = np.empty(chk, dtype=np.int32)
        fast_sl_chk = np.empty(chk, dtype=np.int32)
        fast_ret_chk = np.empty(chk, dtype=np.float32)
        fast_cnt_chk = np.empty(chk, dtype=np.int32)

        evaluate_best_tp_sl_trade_list_slow(
            top_si[:chk],
            top_ej[:chk],
            sma_trade_T,
            ema_trade_T,
            sig_entry_exec_idx,
            exec_open,
            exec_close,
            np.int32(T_exec),
            hit_long_tp,
            hit_long_sl,
            hit_short_tp,
            hit_short_sl,
            long_tp_eq,
            long_sl_eq,
            short_tp_eq,
            short_sl_eq,
            float(fee_rate),
            np.int8(close_on_end),
            slow_tp_chk,
            slow_sl_chk,
            slow_ret_chk,
            slow_cnt_chk,
        )

        grid_fast_kernel(
            top_si[:chk],
            top_ej[:chk],
            sma_trade_T,
            ema_trade_T,
            sig_entry_exec_idx,
            exec_open,
            exec_close,
            np.int32(T_exec),
            hit_long_tp,
            hit_long_sl,
            hit_short_tp,
            hit_short_sl,
            long_tp_eq,
            long_sl_eq,
            short_tp_eq,
            short_sl_eq,
            trade_counts[:chk],
            log_fac_tp_long,
            log_fac_sl_long,
            log_fac_tp_short,
            log_fac_sl_short,
            float(log_fee_two_sides),
            log_exec_open,
            float(log_last_close),
            float(fee_two_sides),
            np.int8(close_on_end),
            np.int8(1),
            fast_tp_chk,
            fast_sl_chk,
            fast_ret_chk,
            fast_cnt_chk,
        )

        if not np.array_equal(slow_tp_chk, fast_tp_chk):
            raise AssertionError("Fast TP index mismatch vs slow reference on self-check subset.")
        if not np.array_equal(slow_sl_chk, fast_sl_chk):
            raise AssertionError("Fast SL index mismatch vs slow reference on self-check subset.")
        if not np.array_equal(fast_cnt_chk, trade_counts[:chk]):
            raise AssertionError("Fast trade counts differ from precomputed trade_counts on self-check subset.")

        ret_err = float(np.max(np.abs(slow_ret_chk.astype(np.float64) - fast_ret_chk.astype(np.float64))))
        ret_tol = 3e-4 if GRID_DIFF_F32 else 1e-4
        if ret_err > ret_tol:
            raise AssertionError(
                f"Fast best_ret mismatch vs slow reference (max diff={ret_err}, tol={ret_tol})."
            )

        print(
            "\nGrid self-check:",
            "passed for",
            chk,
            "strategies; max |best_ret diff| =",
            ret_err,
            "(tol:",
            ret_tol,
            ")",
        )

old_grid_time = math.nan
if GRID_BENCHMARK_SLOW_FULL:
    print("\nGrid-search (slow reference) ...")
    old_tp = np.empty(K, dtype=np.int32)
    old_sl = np.empty(K, dtype=np.int32)
    old_ret = np.empty(K, dtype=np.float32)
    old_cnt = np.empty(K, dtype=np.int32)

    t_old0 = time.perf_counter()
    evaluate_best_tp_sl_trade_list_slow(
        top_si,
        top_ej,
        sma_trade_T,
        ema_trade_T,
        sig_entry_exec_idx,
        exec_open,
        exec_close,
        np.int32(T_exec),
        hit_long_tp,
        hit_long_sl,
        hit_short_tp,
        hit_short_sl,
        long_tp_eq,
        long_sl_eq,
        short_tp_eq,
        short_sl_eq,
        float(fee_rate),
        np.int8(close_on_end),
        old_tp,
        old_sl,
        old_ret,
        old_cnt,
    )
    t_old1 = time.perf_counter()
    old_grid_time = t_old1 - t_old0
    print("  slow grid-search time (s):", round(old_grid_time, 3))

print("\nGrid-search (fast monotone-only boundary) ...")
t_grid0 = time.perf_counter()
grid_fast_kernel(
    top_si,
    top_ej,
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    exec_open,
    exec_close,
    np.int32(T_exec),
    hit_long_tp,
    hit_long_sl,
    hit_short_tp,
    hit_short_sl,
    long_tp_eq,
    long_sl_eq,
    short_tp_eq,
    short_sl_eq,
    trade_counts,
    log_fac_tp_long,
    log_fac_sl_long,
    log_fac_tp_short,
    log_fac_sl_short,
    float(log_fee_two_sides),
    log_exec_open,
    float(log_last_close),
    float(fee_two_sides),
    np.int8(close_on_end),
    np.int8(0),
    best_tp_idx,
    best_sl_idx,
    best_ret,
    grid_trade_counts,
)
t_grid1 = time.perf_counter()
fast_grid_time = t_grid1 - t_grid0

total_confirms = np.int64(confirm_counts_selected.sum())
total_trades = np.int64(grid_trade_counts.sum())
est_old_updates = float(total_confirms) * float(G)
est_new_updates = float(total_trades) * float(G)

print("  fast grid-search time (s):", round(fast_grid_time, 3))
if not math.isnan(old_grid_time) and fast_grid_time > 0.0:
    print("  speedup vs slow:", round(old_grid_time / fast_grid_time, 2), "x")
print("  estimated old updates (confirmations x grid):", int(est_old_updates))
print("  estimated new updates (trades x grid):", int(est_new_updates))
if est_new_updates > 0.0:
    print("  estimated reduction factor:", round(est_old_updates / est_new_updates, 2), "x")
else:
    print("  estimated reduction factor: inf (no trades)")

print(
    "  sample best cell:",
    int(best_tp_idx[0]),
    int(best_sl_idx[0]),
    float(best_ret[0] * 100.0),
    "%",
)


# -------------------------
# Metrics computation and final ranking
# -------------------------
@njit_cached(parallel=True, fastmath=True)
def metrics_for_best_cell_trade_list(
    pair_sma_idx: np.ndarray,
    pair_ema_idx: np.ndarray,
    sma_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open: np.ndarray,
    exec_close: np.ndarray,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    fee_rate: float,
    close_on_end: np.int8,
    bars_per_year_exec: float,
    best_tp_idx: np.ndarray,
    best_sl_idx: np.ndarray,
    out_total_ret: np.ndarray,
    out_max_dd: np.ndarray,
    out_sharpe: np.ndarray,
    out_trades: np.ndarray,
    out_winrate: np.ndarray,
    out_avg_trade_ret: np.ndarray,
    out_avg_trade_bars: np.ndarray,
    out_exposure: np.ndarray,
) -> None:
    """Compute full metrics for each strategy at its selected best TP/SL cell.

    Parameters:
        pair_sma_idx: Selected SMA indices.
        pair_ema_idx: Selected EMA indices.
        sma_trade_T: 1h SMA signal matrix for selected rows.
        ema_trade_T: 1h EMA signal matrix for selected rows.
        sig_entry_exec_idx: 1h-to-execution entry mapping.
        exec_open: Execution-bar open prices.
        exec_close: Execution-bar close prices.
        T_exec: Number of execution bars.
        hit_*: Hit-time tables.
        *_eq: TP/SL level factors.
        fee_rate: Fee per side.
        close_on_end: Whether to close at end.
        bars_per_year_exec: Annualization denominator based on execution TF.
        best_tp_idx: Best TP index per strategy.
        best_sl_idx: Best SL index per strategy.
        out_*: Output metric arrays.

    Side effects:
        Writes strategy metrics to output arrays.
    """
    K_ = pair_sma_idx.shape[0]
    n_sig = sma_trade_T.shape[1]

    one_minus_fee = 1.0 - fee_rate
    fee_two_sides = one_minus_fee * one_minus_fee
    total_exec_bars = float(T_exec)
    years = total_exec_bars / bars_per_year_exec if bars_per_year_exec > 0.0 else 1.0
    if years <= 0.0:
        years = 1.0

    last_close = float(exec_close[T_exec - 1])

    for k in nb.prange(K_):
        si = pair_sma_idx[k]
        ej = pair_ema_idx[k]

        tp_i = np.int32(best_tp_idx[k])
        sl_i = np.int32(best_sl_idx[k])

        entry_arr = np.empty(n_sig, dtype=np.int32)
        dir_arr = np.empty(n_sig, dtype=np.int8)
        sig_exit_arr = np.empty(n_sig, dtype=np.int32)

        n_trades = build_trade_list_for_pair(
            sma_trade_T[si],
            ema_trade_T[ej],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )

        eq = 1.0
        peak = 1.0
        max_dd = 0.0

        trade_cnt = 0
        win = 0
        sum_tr = 0.0
        sum_tr2 = 0.0
        sum_bars = 0.0
        exposure_bars = 0.0

        for tr in range(n_trades):
            pf, exit_exec, closed = evaluate_trade_factor(
                dir_arr[tr],
                entry_arr[tr],
                sig_exit_arr[tr],
                tp_i,
                sl_i,
                exec_open,
                last_close,
                T_exec,
                close_on_end,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                long_tp_eq,
                long_sl_eq,
                short_tp_eq,
                short_sl_eq,
            )

            if closed == 1:
                fac = fee_two_sides * pf
                eq *= fac

                tr_ret = fac - 1.0
                trade_cnt += 1
                if tr_ret > 0.0:
                    win += 1
                sum_tr += tr_ret
                sum_tr2 += tr_ret * tr_ret

                bars_held = float(exit_exec - entry_arr[tr])
                if bars_held < 0.0:
                    bars_held = 0.0
                sum_bars += bars_held
                exposure_bars += bars_held

                if eq > peak:
                    peak = eq
                dd = (eq / peak) - 1.0
                if dd < max_dd:
                    max_dd = dd

        total_ret = eq - 1.0

        sh = 0.0
        if trade_cnt > 1:
            mean_tr = sum_tr / trade_cnt
            var_tr = (sum_tr2 / trade_cnt) - (mean_tr * mean_tr)
            if var_tr > 1e-18:
                trades_per_year = trade_cnt / years
                sh = (mean_tr / math.sqrt(var_tr)) * math.sqrt(trades_per_year)

        out_total_ret[k] = np.float32(total_ret)
        out_max_dd[k] = np.float32(max_dd)
        out_sharpe[k] = np.float32(sh)
        out_trades[k] = np.int32(trade_cnt)

        if trade_cnt > 0:
            out_winrate[k] = np.float32(win / trade_cnt)
            out_avg_trade_ret[k] = np.float32(sum_tr / trade_cnt)
            out_avg_trade_bars[k] = np.float32(sum_bars / trade_cnt)
        else:
            out_winrate[k] = np.float32(0.0)
            out_avg_trade_ret[k] = np.float32(0.0)
            out_avg_trade_bars[k] = np.float32(0.0)

        if total_exec_bars > 0.0:
            out_exposure[k] = np.float32(exposure_bars / total_exec_bars)
        else:
            out_exposure[k] = np.float32(0.0)


out_total = np.empty(K, dtype=np.float32)
out_dd = np.empty(K, dtype=np.float32)
out_sh = np.empty(K, dtype=np.float32)
out_tr = np.empty(K, dtype=np.int32)
out_wr = np.empty(K, dtype=np.float32)
out_atr = np.empty(K, dtype=np.float32)
out_atb = np.empty(K, dtype=np.float32)
out_ex = np.empty(K, dtype=np.float32)

# Warmup metrics kernel
metrics_for_best_cell_trade_list(
    top_si[:w],
    top_ej[:w],
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    exec_open,
    exec_close,
    np.int32(T_exec),
    hit_long_tp,
    hit_long_sl,
    hit_short_tp,
    hit_short_sl,
    long_tp_eq,
    long_sl_eq,
    short_tp_eq,
    short_sl_eq,
    float(fee_rate),
    np.int8(close_on_end),
    float(bars_per_year_exec),
    best_tp_idx[:w],
    best_sl_idx[:w],
    out_total[:w],
    out_dd[:w],
    out_sh[:w],
    out_tr[:w],
    out_wr[:w],
    out_atr[:w],
    out_atb[:w],
    out_ex[:w],
)

print("\nMetrics for best TP/SL ...")
t_met0 = time.perf_counter()
metrics_for_best_cell_trade_list(
    top_si,
    top_ej,
    sma_trade_T,
    ema_trade_T,
    sig_entry_exec_idx,
    exec_open,
    exec_close,
    np.int32(T_exec),
    hit_long_tp,
    hit_long_sl,
    hit_short_tp,
    hit_short_sl,
    long_tp_eq,
    long_sl_eq,
    short_tp_eq,
    short_sl_eq,
    float(fee_rate),
    np.int8(close_on_end),
    float(bars_per_year_exec),
    best_tp_idx,
    best_sl_idx,
    out_total,
    out_dd,
    out_sh,
    out_tr,
    out_wr,
    out_atr,
    out_atb,
    out_ex,
)
t_met1 = time.perf_counter()
print("  metrics time (s):", round(t_met1 - t_met0, 3))


# -------------------------
# Final ranking
# -------------------------
def aligned_to_step(values: np.ndarray, start: float, step: float, tol: float = 1e-6) -> np.ndarray:
    """Check if grid values align to the configured arithmetic step.

    Parameters:
        values: Values to validate.
        start: Grid start value.
        step: Grid step value.
        tol: Absolute tolerance on rounding residual.

    Returns:
        Boolean mask with alignment result per value.
    """
    scaled = (values - start) / step
    return np.abs(scaled - np.round(scaled)) <= tol


res = pd.DataFrame(
    {
        "return_pct": out_total.astype(np.float64) * 100.0,
        "max_dd_pct": out_dd.astype(np.float64) * 100.0,
        "sharpe": out_sh.astype(np.float64),
        "trades": out_tr.astype(np.int64),
        "winrate_pct": out_wr.astype(np.float64) * 100.0,
        "exposure_pct": out_ex.astype(np.float64) * 100.0,
        "avg_trade_ret_pct": out_atr.astype(np.float64) * 100.0,
        "avg_trade_exec_bars": out_atb.astype(np.float64),
        "sma_i": top_si.astype(np.int64),
        "ema_i": top_ej.astype(np.int64),
        "best_tp_idx": best_tp_idx.astype(np.int64),
        "best_sl_idx": best_sl_idx.astype(np.int64),
        "best_tp_pct": tp_grid[best_tp_idx].astype(np.float64),
        "best_sl_pct": sl_grid[best_sl_idx].astype(np.float64),
        "signal_tf_intent": signal_tf,
        "exec_tf_intent": exec_tf,
        "effective_signal_tf": effective_signal_tf,
        "effective_exec_tf": effective_exec_tf,
        "signal_tf_detected": effective_signal_tf,
        "exec_tf_detected": effective_exec_tf,
        "detected_exec_dt_ms": detected_dt_ms,
        "detected_signal_dt_ms": int(detected_signal_dt_ms),
        "detected_dt_ms": detected_dt_ms,
        "tp_start_pct": tp_start_pct,
        "tp_stop_pct": tp_stop_pct,
        "tp_step_pct": tp_step_pct,
        "sl_start_pct": sl_start_pct,
        "sl_stop_pct": sl_stop_pct,
        "sl_step_pct": sl_step_pct,
        "confirm_cnt": n_confirm[top_si, top_ej].astype(np.float64),
        "trade_list_cnt": grid_trade_counts.astype(np.int64),
        "proxy_adj": proxy_adj[top_si, top_ej].astype(np.float64),
    }
)

res = res.sort_values(["return_pct", "sharpe"], ascending=[False, False], kind="mergesort").reset_index(drop=True)

# Sanity checks
if not np.all((res["best_tp_pct"].to_numpy() >= tp_start_pct) & (res["best_tp_pct"].to_numpy() <= tp_stop_pct + 1e-9)):
    raise AssertionError("best_tp_pct is out of configured range.")
if not np.all((res["best_sl_pct"].to_numpy() >= sl_start_pct) & (res["best_sl_pct"].to_numpy() <= sl_stop_pct + 1e-9)):
    raise AssertionError("best_sl_pct is out of configured range.")
if not np.all(aligned_to_step(res["best_tp_pct"].to_numpy(), tp_start_pct, tp_step_pct)):
    raise AssertionError("best_tp_pct is not aligned to tp_step_pct.")
if not np.all(aligned_to_step(res["best_sl_pct"].to_numpy(), sl_start_pct, sl_step_pct)):
    raise AssertionError("best_sl_pct is not aligned to sl_step_pct.")

key_cols = ["return_pct", "trades", "winrate_pct", "best_tp_pct", "best_sl_pct"]
if res[key_cols].isna().any().any():
    raise AssertionError("NaNs detected in key result columns.")

TOP_N = 50
top = res.head(TOP_N).copy()
top["sma_col"] = sma_cols_keep[top["sma_i"].to_numpy()]
top["ema_col"] = ema_cols_keep[top["ema_i"].to_numpy()]

top = top[
    [
        "return_pct",
        "max_dd_pct",
        "sharpe",
        "trades",
        "winrate_pct",
        "best_tp_pct",
        "best_sl_pct",
        "tp_step_pct",
        "sl_step_pct",
        "sma_col",
        "ema_col",
        "confirm_cnt",
        "trade_list_cnt",
        "proxy_adj",
    ]
]

print("\nTOP:")
try:
    display(top)
except NameError:
    print(top.to_string(index=False))














BASE_DIR: /Users/daniildegtyarev/Projects/roehub.com/tests/notebook_tests/precompute/btcusdt_5m
NPY: /Users/daniildegtyarev/Projects/roehub.com/tests/notebook_tests/precompute/btcusdt_5m/prices_and_signals_5m.npy
HIT files exist: True True True True

Config:
  signal_tf: 1h exec_tf: 5m
  fee_rate: 0.0004 close_on_end: 1
  E: top_frac_side: 0.31622776601683794 min_nonzero_single: 200
  A/B/D: min_confirm: 30 top_frac_pairs: 0.1 time_chunk: 4096
  TP grid: (4.0, 50.0, 0.5)
  SL grid: (2.0, 25.0, 0.5)
Loaded columns from: /Users/daniildegtyarev/Projects/roehub.com/tests/notebook_tests/precompute/btcusdt_5m/prices_and_signals_5m_columns.npy

Data shape: (73285, 2359)
SMA cols: 1176 EMA cols: 1176

Hit-time loaded:
  long_tp: (93, 73285) uint32
  long_sl: (47, 73285) uint32
  short_tp: (93, 73285) uint32
  short_sl: (47, 73285) uint32
  TP levels: 93 SL levels: 47 grid cells: 4371
  bars_per_year_exec: 8760.0

Timeframe reality check:
  signal_tf_intent: 1h
  exec_tf_intent: 5m
  detected_e

A/B precompute (1h chunked GEMM):   0%|          | 0/18 [00:00<?, ?it/s]


A/B/D:
  valid pairs: 138384 / 138384
  selected strategies: 13839

Mapping 1h -> execution indices:
  sig bars: 73285
  first entries: [1, 2, 3, 4, 5]
  last entries: [73281, 73282, 73283, 73284, 73285]

Numba threads: 11

Trade list construction:
  mean confirmations per strategy: 58405.60018787484
  mean trades per strategy: 1289.5114531396778
  median confirmations per strategy: 58904.0
  median trades per strategy: 1238.0
  trade-count kernel time (s): 0.19

Sanity & Diagnostics:
  detected_exec_dt_ms: 3600000
  detected_signal_dt_ms: 3600000
  detected_exec_tf: 1h
  detected_signal_tf: 1h
  effective_exec_tf: 1h
  signal bars: 73285
  execution bars: 73285
  exec_dt_min_ms: 3600000 exec_dt_max_ms: 118800000
  signal_dt_min_ms: 3600000 signal_dt_max_ms: 118800000
  mean confirmations per strategy: 58405.60018787484
  mean trades per strategy: 1289.5114531396778

Grid debug toggles: self_check= False self_check_n= 50 bench_slow_full= False diff_f32= False

Grid-search (fast monoto

,return_pct,max_dd_pct,sharpe,trades,winrate_pct,best_tp_pct,best_sl_pct,tp_step_pct,sl_step_pct,sma_col,ema_col,confirm_cnt,trade_list_cnt,proxy_adj
0,23104.066467,-53.115642,1.179233,1179,24.851570,39.5,15.0,0.5,0.5,signal|ma.sma|variant=846|source=high|window=67,signal|ma.ema|variant=979|source=high|window=200,58256.0,1179,-30.122627
1,23091.500854,-59.881026,1.198613,1161,24.375539,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1055|source=open|window=80,signal|ma.ema|variant=970|source=high|window=191,60599.0,1161,-31.229156
2,22893.168640,-57.260215,1.196826,1171,24.423569,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1049|source=open|window=74,signal|ma.ema|variant=970|source=high|window=191,59740.0,1171,-30.681772
3,22876.516724,-58.976972,1.197209,1167,24.250214,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1053|source=open|window=78,signal|ma.ema|variant=970|source=high|window=191,60329.0,1167,-31.102097
4,22609.471130,-60.190451,1.195168,1165,24.291846,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1054|source=open|window=79,signal|ma.ema|variant=970|source=high|window=191,60484.0,1165,-31.093121
5,22581.738281,-59.875160,1.192416,1157,24.114089,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1055|source=open|window=80,signal|ma.ema|variant=971|source=high|window=192,60552.0,1157,-31.291557
6,22563.076782,-55.467105,1.194085,1133,24.360105,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1049|source=open|window=74,signal|ma.ema|variant=979|source=high|window=200,59307.0,1133,-30.431183
7,22482.667542,-52.976114,1.174867,1193,24.979044,39.5,15.0,0.5,0.5,signal|ma.sma|variant=845|source=high|window=66,signal|ma.ema|variant=979|source=high|window=200,58127.0,1193,-30.046740
8,22405.273438,-56.012189,1.193430,1127,24.134871,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1053|source=open|window=78,signal|ma.ema|variant=979|source=high|window=200,59888.0,1127,-30.846699
9,22340.132141,-57.353014,1.193128,1167,24.250214,40.0,2.5,0.5,0.5,signal|ma.sma|variant=1052|source=open|window=77,signal|ma.ema|variant=970|source=high|window=191,60178.0,1167,-31.176706
